In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/latest.pt
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/log.jsonl
/kaggle/input/models/achiver7/arch1-modeldecoder/transformers/default/1/run_arch1_real/best.pt
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/latest.pt
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/log.jsonl
/kaggle/input/models/achiver7/arch2-model/transformers/default/1/run_arch2_preln_efficient/best.pt
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/latest.pt
/kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/latest.pt
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_swa_v2/log.jsonl
/kaggle/input/datasets/achiver7/arch3-model/run_arch3_

In [2]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("HF_TOKEN")
p = snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset",
                      allow_patterns=["Phase_2_LNER/*"], token=tok, local_dir="/kaggle/working/ner_data")
print(p)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/kaggle/working/ner_data


In [3]:
from pathlib import Path

NER_ROOT = Path("/kaggle/working/ner_data/Phase_2_LNER")

for p in sorted(NER_ROOT.rglob("*")):
    if p.is_file():
        print(p)

/kaggle/working/ner_data/Phase_2_LNER/Test/fold3-text.json
/kaggle/working/ner_data/Phase_2_LNER/Train/fold1-gold.json
/kaggle/working/ner_data/Phase_2_LNER/Train/fold1-text.json
/kaggle/working/ner_data/Phase_2_LNER/Train/fold2-gold.json
/kaggle/working/ner_data/Phase_2_LNER/Train/fold2-text.json
/kaggle/working/ner_data/Phase_2_LNER/Train/labels.txt


Phase 2 dataset

In [4]:
from pathlib import Path

NER_ROOT = Path("/kaggle/working/ner_data/Phase_2_LNER")

FOLD1_TEXT  = NER_ROOT / "Train" / "fold1-text.json"
FOLD1_GOLD  = NER_ROOT / "Train" / "fold1-gold.json"

FOLD2_TEXT  = NER_ROOT / "Train" / "fold2-text.json"
FOLD2_GOLD  = NER_ROOT / "Train" / "fold2-gold.json"

LABELS_FILE = NER_ROOT / "Train" / "labels.txt"

FOLD3_TEXT  = NER_ROOT / "Test" / "fold3-text.json"

print("Phase-2 dataset:")
for p in [
    FOLD1_TEXT, FOLD1_GOLD,
    FOLD2_TEXT, FOLD2_GOLD,
    LABELS_FILE, FOLD3_TEXT
]:
    print(f"{p.exists():5}  {p}")

Phase-2 dataset:
    1  /kaggle/working/ner_data/Phase_2_LNER/Train/fold1-text.json
    1  /kaggle/working/ner_data/Phase_2_LNER/Train/fold1-gold.json
    1  /kaggle/working/ner_data/Phase_2_LNER/Train/fold2-text.json
    1  /kaggle/working/ner_data/Phase_2_LNER/Train/fold2-gold.json
    1  /kaggle/working/ner_data/Phase_2_LNER/Train/labels.txt
    1  /kaggle/working/ner_data/Phase_2_LNER/Test/fold3-text.json


In [5]:
NER_ROOT = "/kaggle/working/ner_data/Phase_2_LNER"

In [6]:
FOLD1_TEXT  = f"{NER_ROOT}/Train/fold1-text.json"
FOLD1_GOLD  = f"{NER_ROOT}/Train/fold1-gold.json"
FOLD2_TEXT  = f"{NER_ROOT}/Train/fold2-text.json"
FOLD2_GOLD  = f"{NER_ROOT}/Train/fold2-gold.json"
LABELS_FILE = f"{NER_ROOT}/Train/labels.txt"
FOLD3_TEXT  = f"{NER_ROOT}/Test/fold3-text.json"

In [7]:
import json

with open(LABELS_FILE, "r") as f:
    labels = [x.strip() for x in f if x.strip()]

print("Labels:", labels)
print("Number of labels:", len(labels))

with open(FOLD1_TEXT, "r") as f:
    fold1_text = json.load(f)

with open(FOLD1_GOLD, "r") as f:
    fold1_gold = json.load(f)

print("Fold1 documents:", len(fold1_text))
print("Fold1 gold documents:", len(fold1_gold))

doc_id = next(iter(fold1_text))

print("\nExample document ID:", doc_id)
print("Text length:", len(fold1_text[doc_id]))
print("Gold entities:", fold1_gold.get(doc_id, [])[:5])

Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Number of labels: 12
Fold1 documents: 35
Fold1 gold documents: 35

Example document ID: 115651329
Text length: 18616
Gold entities: [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana'], [359, 369, 'COURT', 'High Court'], [575, 585, 'DATE', '18.11.1994']]


In [8]:
import json

# Load Fold 2
with open(FOLD2_TEXT, "r") as f:
    fold2_text = json.load(f)

with open(FOLD2_GOLD, "r") as f:
    fold2_gold = json.load(f)

# Load test
with open(FOLD3_TEXT, "r") as f:
    fold3_text = json.load(f)

print("=" * 60)
print("PHASE-2 DATASET CHECK")
print("=" * 60)

print(f"Fold 1 text docs : {len(fold1_text)}")
print(f"Fold 1 gold docs : {len(fold1_gold)}")
print(f"Fold 2 text docs : {len(fold2_text)}")
print(f"Fold 2 gold docs : {len(fold2_gold)}")
print(f"Test docs        : {len(fold3_text)}")
print(f"Labels           : {len(labels)}")

# Check IDs
print("\nID consistency:")

print(
    "Fold 1:",
    set(fold1_text.keys()) == set(fold1_gold.keys())
)

print(
    "Fold 2:",
    set(fold2_text.keys()) == set(fold2_gold.keys())
)

print("\nExpected:")
print("  Training docs :", len(fold1_text) + len(fold2_text))
print("  Test docs     :", len(fold3_text))

# Basic gold-span validation
bad = []

valid_labels = set(labels)

for split_name, gold in [
    ("fold1", fold1_gold),
    ("fold2", fold2_gold),
]:
    for doc_id, entities in gold.items():
        text = (fold1_text if split_name == "fold1" else fold2_text)[doc_id]

        for ent in entities:
            if len(ent) < 3:
                bad.append((split_name, doc_id, ent, "too_short"))
                continue

            start, end, label = ent[:3]

            if not (0 <= start < end <= len(text)):
                bad.append((split_name, doc_id, ent, "bad_offset"))

            if label not in valid_labels:
                bad.append((split_name, doc_id, ent, "bad_label"))

            # Verify provided entity text if present
            if len(ent) >= 4:
                entity_text = ent[3]
                actual = text[start:end]

                if actual != entity_text:
                    bad.append(
                        (split_name, doc_id, ent, "text_mismatch")
                    )

print("\nGold validation errors:", len(bad))

if bad:
    print("\nFirst 10 errors:")
    for x in bad[:10]:
        print(x)
else:
    print("All gold spans passed basic validation.")

PHASE-2 DATASET CHECK
Fold 1 text docs : 35
Fold 1 gold docs : 35
Fold 2 text docs : 35
Fold 2 gold docs : 35
Test docs        : 35
Labels           : 12

ID consistency:
Fold 1: True
Fold 2: True

Expected:
  Training docs : 70
  Test docs     : 35

Gold validation errors: 0
All gold spans passed basic validation.


In [9]:
import torch
from pathlib import Path

ARCH4_CKPT = Path(
    "/kaggle/input/models/achiversx/"
    "arch4-hybrid-gqa-201m/transformers/default/1/"
    "arch4_hybrid_gqa_201m/best.pt"
)

print("Checkpoint exists:", ARCH4_CKPT.exists())
print("Checkpoint:", ARCH4_CKPT)

ckpt = torch.load(ARCH4_CKPT, map_location="cpu")

print("\nCheckpoint type:", type(ckpt))

if isinstance(ckpt, dict):
    print("Checkpoint keys:", list(ckpt.keys()))

    if "step" in ckpt:
        print("Training step:", ckpt["step"])

    if "best_val" in ckpt:
        print("Best validation:", ckpt["best_val"])

    if "cfg" in ckpt:
        print("\nSaved config:")
        print(ckpt["cfg"])

    state = ckpt["model"] if "model" in ckpt else ckpt

    print("\nState dict tensors:", len(state))

    total_params = sum(
        v.numel()
        for v in state.values()
        if torch.is_tensor(v)
    )

    print("Checkpoint parameters:", f"{total_params:,}")

Checkpoint exists: True
Checkpoint: /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt

Checkpoint type: <class 'dict'>
Checkpoint keys: ['step', 'model', 'best_val', 'cfg']
Training step: 10000
Best validation: 2.475979009747505

Saved config:
{'DATA_DIR': '/kaggle/input/datasets/achiver07/phase1-tokens-v40k-clean1', 'VOCAB_SIZE': 40000, 'D_MODEL': 1024, 'N_LAYERS': 12, 'N_Q_HEADS': 16, 'N_KV_HEADS': 8, 'D_FF': 3328, 'MAX_SEQ_LEN': 1024, 'SEQ_LEN': 1024, 'WINDOW': 256, 'GLOBAL_EVERY': 3, 'ROPE_BASE': 10000.0, 'DROPOUT': 0.0, 'INIT_STD': 0.02, 'TOKENS_PER_STEP': 16384, 'CANDIDATES': [(8, False), (4, False), (8, True), (4, True), (2, True)], 'MAX_STEPS': 10000, 'LR': 0.0003, 'MIN_LR_RATIO': 0.1, 'WARMUP_STEPS': 500, 'WEIGHT_DECAY': 0.1, 'GRAD_CLIP': 1.0, 'LOSS_CHUNK': 2048, 'USE_FUSED_ADAMW': True, 'USE_COMPILE': True, 'COMPILE_MODE': 'default', 'EVAL_EVERY': 500, 'EVAL_BATCHES_PER_SOURCE': 10, 'SAVE_EVERY': 500, 'LOG_EVERY': 50, 'T

In [10]:
# ============================================================
# ARCH4 PHASE-2 BACKBONE PREFLIGHT
# ============================================================

import torch
import torch.nn as nn

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 1. Check checkpoint parameter breakdown
# ------------------------------------------------------------

state = ckpt["model"]

embedding_params = 0
other_params = 0

for name, tensor in state.items():
    if not torch.is_tensor(tensor):
        continue

    n = tensor.numel()

    if "token_emb" in name or "embedding" in name:
        embedding_params += n
    else:
        other_params += n

print("\nParameter breakdown:")
print(f"Embedding params : {embedding_params:,}")
print(f"Other params     : {other_params:,}")
print(f"Total            : {embedding_params + other_params:,}")


# ------------------------------------------------------------
# 2. Show embedding-related checkpoint keys
# ------------------------------------------------------------

print("\nEmbedding-related keys:")

for name, tensor in state.items():
    if "emb" in name.lower():
        print(
            f"{name:50s}",
            tuple(tensor.shape),
            f"{tensor.numel():,}"
        )

Device: cuda
GPU: Tesla T4

Parameter breakdown:
Embedding params : 40,960,000
Other params     : 201,443,328
Total            : 242,403,328

Embedding-related keys:
token_embedding.weight                             (40000, 1024) 40,960,000


## Arch4 Checkpoint Parameter Verification

The Arch4 Phase-1 checkpoint was successfully inspected on a Tesla T4 GPU.

| Component | Parameters |
|---|---:|
| Token embedding | 40,960,000 |
| Transformer core | 201,443,328 |
| **Total** | **242,403,328** |

The token embedding matrix has shape:

```text
(40,000, 1,024)


### Next step

Now we should do the **actual backbone forward-pass test**:

```text
Phase-1 checkpoint
       ↓
Arch4 backbone
       ↓
hidden_states()
       ↓
[B, 1024, 1024]
       ↓
13-class NER head
       ↓
[B, 1024, 13]

In [11]:
# ============================================================
# ARCH4 — ORIGINAL MODEL DEFINITIONS
# Phase-2 integration only. No training in this cell.
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# 1. Rotary positional embedding utilities
# ------------------------------------------------------------

def rotate_half(x):
    half = x.shape[-1] // 2
    return torch.cat([-x[..., half:], x[..., :half]], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, dim, 2, dtype=torch.float32) / dim
            )
        )

        frequencies = torch.outer(
            torch.arange(max_seq_len, dtype=torch.float32),
            inv_freq
        )

        embeddings = torch.cat(
            [frequencies, frequencies],
            dim=-1
        )

        self.register_buffer(
            "cos",
            embeddings.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin",
            embeddings.sin(),
            persistent=False
        )

    def forward(self, T):
        return (
            self.cos[:T][None, None],
            self.sin[:T][None, None]
        )


# ------------------------------------------------------------
# 2. SwiGLU feed-forward network
# ------------------------------------------------------------

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.in_proj = nn.Linear(
            d_model,
            2 * d_ff,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False
        )

    def forward(self, x):
        gate, value = self.in_proj(x).chunk(2, dim=-1)

        return self.out_proj(
            F.silu(gate) * value
        )


# ------------------------------------------------------------
# 3. Grouped-query attention
# ------------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        d_model,
        n_q,
        n_kv,
        window,
        is_global,
        dropout=0.0
    ):
        super().__init__()

        assert d_model % n_q == 0
        assert n_q % n_kv == 0

        self.n_q = n_q
        self.n_kv = n_kv

        self.head_dim = d_model // n_q
        self.rep = n_q // n_kv

        self.window = window
        self.is_global = is_global
        self.dropout = dropout

        q_dim = n_q * self.head_dim
        kv_dim = n_kv * self.head_dim

        self.split = [q_dim, kv_dim, kv_dim]

        self.qkv_proj = nn.Linear(
            d_model,
            q_dim + 2 * kv_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False
        )

    def forward(self, x, cos, sin, mask):
        B, T, C = x.shape

        Hq = self.n_q
        Hkv = self.n_kv
        Dh = self.head_dim
        rep = self.rep

        q, k, v = torch.split(
            self.qkv_proj(x),
            self.split,
            dim=-1
        )

        q = q.view(B, T, Hq, Dh).transpose(1, 2)
        k = k.view(B, T, Hkv, Dh).transpose(1, 2)
        v = v.view(B, T, Hkv, Dh).transpose(1, 2)

        cos = cos.to(q.dtype)
        sin = sin.to(q.dtype)

        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin

        dropout_p = self.dropout if self.training else 0.0

        # Global causal attention
        if self.is_global:
            k = (
                k[:, :, None]
                .expand(B, Hkv, rep, T, Dh)
                .reshape(B, Hq, T, Dh)
            )

            v = (
                v[:, :, None]
                .expand(B, Hkv, rep, T, Dh)
                .reshape(B, Hq, T, Dh)
            )

            y = F.scaled_dot_product_attention(
                q,
                k,
                v,
                dropout_p=dropout_p,
                is_causal=True
            )

            return self.out_proj(
                y.transpose(1, 2).reshape(B, T, C)
            )

        # Local chunked attention
        W = self.window
        N = T // W

        qc = (
            q.view(B, Hq, N, W, Dh)
            .transpose(1, 2)
            .reshape(B * N, Hq, W, Dh)
        )

        def chunk_kv(t):
            tc = t.view(B, Hkv, N, W, Dh)

            previous = F.pad(
                tc[:, :, :-1],
                (0, 0, 0, 0, 1, 0)
            )

            combined = torch.cat(
                [previous, tc],
                dim=3
            )

            combined = combined.transpose(1, 2)[:, :, :, None]

            return (
                combined
                .expand(B, N, Hkv, rep, 2 * W, Dh)
                .reshape(B * N, Hq, 2 * W, Dh)
            )

        y = F.scaled_dot_product_attention(
            qc,
            chunk_kv(k),
            chunk_kv(v),
            attn_mask=mask,
            dropout_p=dropout_p
        )

        y = (
            y.view(B, N, Hq, W, Dh)
            .permute(0, 1, 3, 2, 4)
            .reshape(B, T, C)
        )

        return self.out_proj(y)


# ------------------------------------------------------------
# 4. Transformer block
# ------------------------------------------------------------

class Arch4Block(nn.Module):
    def __init__(self, cfg, idx):
        super().__init__()

        self.is_global = (
            (idx + 1) % cfg["GLOBAL_EVERY"] == 0
        )

        self.norm1 = nn.LayerNorm(cfg["D_MODEL"])

        self.attn = GQAAttention(
            cfg["D_MODEL"],
            cfg["N_Q_HEADS"],
            cfg["N_KV_HEADS"],
            cfg["WINDOW"],
            self.is_global,
            cfg["DROPOUT"]
        )

        self.norm2 = nn.LayerNorm(cfg["D_MODEL"])

        self.ffn = SwiGLU(
            cfg["D_MODEL"],
            cfg["D_FF"]
        )

    def forward(self, x, cos, sin, mask):
        x = x + self.attn(
            self.norm1(x),
            cos,
            sin,
            mask
        )

        return x + self.ffn(
            self.norm2(x)
        )


# ------------------------------------------------------------
# 5. Original Arch4 backbone
# ------------------------------------------------------------

class Architecture4(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        W = cfg["WINDOW"]
        self.window = W

        assert cfg["MAX_SEQ_LEN"] % W == 0
        assert cfg["SEQ_LEN"] % W == 0

        self.token_embedding = nn.Embedding(
            cfg["VOCAB_SIZE"],
            cfg["D_MODEL"]
        )

        self.layers = nn.ModuleList([
            Arch4Block(cfg, i)
            for i in range(cfg["N_LAYERS"])
        ])

        self.final_norm = nn.LayerNorm(
            cfg["D_MODEL"]
        )

        # Phase-1 language-model head.
        # Its weight is tied to token_embedding.weight.
        self.lm_head = nn.Linear(
            cfg["D_MODEL"],
            cfg["VOCAB_SIZE"],
            bias=False
        )

        self.lm_head.weight = self.token_embedding.weight

        self.rope = RotaryEmbedding(
            cfg["D_MODEL"] // cfg["N_Q_HEADS"],
            cfg["MAX_SEQ_LEN"],
            cfg["ROPE_BASE"]
        )

        # Chunked local-attention mask
        n_chunks = cfg["MAX_SEQ_LEN"] // W

        i = torch.arange(W)[:, None]
        j = torch.arange(2 * W)[None, :]

        mask = (j > i) & (j <= i + W)
        mask = mask[None].repeat(n_chunks, 1, 1)

        mask[0] &= (j >= W)

        self.register_buffer(
            "chunk_mask",
            mask[:, None],
            persistent=False
        )

        self._init_weights()

    def _init_weights(self):
        std = self.cfg["INIT_STD"]

        nn.init.normal_(
            self.token_embedding.weight,
            0.0,
            std
        )

        for module in self.modules():
            if (
                isinstance(module, nn.Linear)
                and module.weight is not self.token_embedding.weight
            ):
                nn.init.normal_(
                    module.weight,
                    0.0,
                    std
                )

        scale = 1.0 / math.sqrt(
            2 * self.cfg["N_LAYERS"]
        )

        for block in self.layers:
            block.attn.out_proj.weight.data.mul_(scale)
            block.ffn.out_proj.weight.data.mul_(scale)

    def hidden_states(self, input_ids):
        B, T = input_ids.shape
        W = self.window

        assert T % W == 0

        N = T // W

        x = self.token_embedding(input_ids)

        cos, sin = self.rope(T)

        mask = (
            self.chunk_mask[:N][None]
            .expand(B, N, 1, W, 2 * W)
            .reshape(B * N, 1, W, 2 * W)
        )

        for block in self.layers:
            x = block(x, cos, sin, mask)

        return self.final_norm(x)

    def forward(self, input_ids):
        return self.lm_head(
            self.hidden_states(input_ids)
        )


print("Arch4 model definitions loaded successfully.")
print("Architecture4 available:", "Architecture4" in globals())

Arch4 model definitions loaded successfully.
Architecture4 available: True


In [12]:
# ============================================================
# PHASE 2 — ARCH4 BACKBONE FORWARD-PASS PREFLIGHT
# ============================================================
# Purpose:
# 1. Load the Phase-1 Arch4 checkpoint.
# 2. Verify that the checkpoint matches Architecture4.
# 3. Run hidden_states() on a 1024-token input.
# 4. Attach a 13-class NER head.
# 5. Verify output shapes and numerical validity.
#
# This cell DOES NOT train the model.
# ============================================================

import json
import torch
import torch.nn as nn
from pathlib import Path
from tokenizers import Tokenizer

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARCH4_CKPT = Path(
    "/kaggle/input/models/achiversx/"
    "arch4-hybrid-gqa-201m/transformers/default/1/"
    "arch4_hybrid_gqa_201m/best.pt"
)

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

NER_ROOT = Path("/kaggle/working/ner_data/Phase_2_LNER")

FOLD1_TEXT = NER_ROOT / "Train" / "fold1-text.json"
LABELS_FILE = NER_ROOT / "Train" / "labels.txt"

MAX_LEN = 1024
D_MODEL = 1024
NUM_CLASSES = 13  # 12 entity labels + O

print("=" * 65)
print("ARCH4 PHASE-2 FORWARD-PASS PREFLIGHT")
print("=" * 65)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Validate required files and architecture definition
# ------------------------------------------------------------

for path in [ARCH4_CKPT, TOKENIZER_PATH, FOLD1_TEXT, LABELS_FILE]:
    assert path.is_file(), f"Required file not found: {path}"

assert "Architecture4" in globals(), (
    "Architecture4 is not defined. Run the original Phase-1 "
    "Architecture4 class-definition cell first."
)

print("\nRequired files: PASS")
print("Architecture4 class: available")

# ------------------------------------------------------------
# 3. Load checkpoint and saved configuration
# ------------------------------------------------------------

ckpt = torch.load(ARCH4_CKPT, map_location="cpu", weights_only=False)

assert isinstance(ckpt, dict), "Unexpected checkpoint format."
assert "model" in ckpt, "Checkpoint does not contain a model state dict."
assert "cfg" in ckpt, "Checkpoint does not contain its configuration."

CFG = dict(ckpt["cfg"])
state_dict = ckpt["model"]

print("\nCheckpoint step:", ckpt.get("step"))
print("Best validation loss:", ckpt.get("best_val"))
print("Vocabulary size:", CFG["VOCAB_SIZE"])
print("Model dimension:", CFG["D_MODEL"])
print("Transformer layers:", CFG["N_LAYERS"])
print("Query heads:", CFG["N_Q_HEADS"])
print("KV heads:", CFG["N_KV_HEADS"])
print("Feed-forward dimension:", CFG["D_FF"])

checkpoint_numel = sum(
    tensor.numel()
    for tensor in state_dict.values()
    if torch.is_tensor(tensor)
)

print("Checkpoint state-dict tensor count:", len(state_dict))
print("Checkpoint state-dict parameter entries:", f"{checkpoint_numel:,}")

assert CFG["VOCAB_SIZE"] == 40000
assert CFG["D_MODEL"] == D_MODEL
assert CFG["MAX_SEQ_LEN"] == MAX_LEN

# ------------------------------------------------------------
# 4. Instantiate the original Phase-1 architecture
# ------------------------------------------------------------
# This assumes the original Architecture4 constructor accepts
# the saved configuration dictionary. If your original class
# uses a different constructor signature, adapt this line to
# match that original implementation.

backbone = Architecture4(CFG)

# Strict loading ensures that missing or unexpected parameters
# cannot silently pass unnoticed.
load_result = backbone.load_state_dict(state_dict, strict=True)

print("\nCheckpoint loading: PASS")
print("Missing keys:", len(load_result.missing_keys))
print("Unexpected keys:", len(load_result.unexpected_keys))

backbone = backbone.to(DEVICE)
backbone.eval()

# ------------------------------------------------------------
# 5. Load tokenizer and a real training document
# ------------------------------------------------------------

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

with open(FOLD1_TEXT, "r", encoding="utf-8") as f:
    fold1_text = json.load(f)

sample_doc_id = next(iter(fold1_text))
sample_text = fold1_text[sample_doc_id]

encoded = tokenizer.encode(sample_text)
token_ids = encoded.ids[:MAX_LEN]

assert len(token_ids) > 0, "Tokenization produced no tokens."
assert max(token_ids) < CFG["VOCAB_SIZE"], (
    "Token ID exceeds the model vocabulary."
)

print("\nSample document ID:", sample_doc_id)
print("Document characters:", len(sample_text))
print("Tokens available:", len(encoded.ids))
print("Tokens used:", len(token_ids))

# Pad to 1024 positions. Padding is only for this shape test.
# No loss is computed and no training takes place.
input_ids = torch.zeros(
    (1, MAX_LEN),
    dtype=torch.long,
    device=DEVICE,
)

input_ids[0, :len(token_ids)] = torch.tensor(
    token_ids,
    dtype=torch.long,
    device=DEVICE,
)

assert input_ids.shape == (1, MAX_LEN)
assert MAX_LEN % CFG["WINDOW"] == 0

# ------------------------------------------------------------
# 6. Create the NER classification head
# ------------------------------------------------------------

with open(LABELS_FILE, "r", encoding="utf-8") as f:
    entity_labels = [line.strip() for line in f if line.strip()]

assert len(entity_labels) == 12, (
    f"Expected 12 entity labels, got {len(entity_labels)}"
)

TAG2ID = {"O": 0}
TAG2ID.update({
    label: idx + 1
    for idx, label in enumerate(entity_labels)
})

ID2TAG = {idx: tag for tag, idx in TAG2ID.items()}

assert len(TAG2ID) == NUM_CLASSES
assert len(ID2TAG) == NUM_CLASSES

ner_head = nn.Linear(D_MODEL, NUM_CLASSES).to(DEVICE)
ner_head.eval()

print("\nNER classes:", NUM_CLASSES)
print("Label mapping:", TAG2ID)

# ------------------------------------------------------------
# 7. Run one forward pass
# ------------------------------------------------------------

with torch.inference_mode():
    hidden_states = backbone.hidden_states(input_ids)

    assert isinstance(hidden_states, torch.Tensor), (
        "hidden_states() did not return a tensor. "
        f"Received: {type(hidden_states)}"
    )

    logits = ner_head(hidden_states)

# ------------------------------------------------------------
# 8. Validate outputs
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FORWARD-PASS RESULTS")
print("=" * 65)

print("Input shape:       ", tuple(input_ids.shape))
print("Hidden-state shape:", tuple(hidden_states.shape))
print("NER logits shape:  ", tuple(logits.shape))
print("Hidden dtype:      ", hidden_states.dtype)
print("Logits dtype:      ", logits.dtype)
print("Hidden device:     ", hidden_states.device)

assert hidden_states.shape == (1, MAX_LEN, D_MODEL), (
    f"Unexpected hidden-state shape: {tuple(hidden_states.shape)}"
)

assert logits.shape == (1, MAX_LEN, NUM_CLASSES), (
    f"Unexpected logits shape: {tuple(logits.shape)}"
)

assert torch.isfinite(hidden_states).all().item(), (
    "Hidden states contain NaN or infinity."
)

assert torch.isfinite(logits).all().item(), (
    "NER logits contain NaN or infinity."
)

print("\n" + "=" * 65)
print("PREFLIGHT STATUS: PASS")
print("=" * 65)

print("Phase-1 checkpoint: loaded")
print("Backbone forward pass: successful")
print("NER classification head: successful")
print("Output dimensions: valid")
print("NaN/Inf check: passed")
print("\nNo model training has been performed.")

ARCH4 PHASE-2 FORWARD-PASS PREFLIGHT
Device: cuda
GPU: Tesla T4

Required files: PASS
Architecture4 class: available

Checkpoint step: 10000
Best validation loss: 2.475979009747505
Vocabulary size: 40000
Model dimension: 1024
Transformer layers: 12
Query heads: 16
KV heads: 8
Feed-forward dimension: 3328
Checkpoint state-dict tensor count: 100
Checkpoint state-dict parameter entries: 242,403,328

Checkpoint loading: PASS
Missing keys: 0
Unexpected keys: 0

Sample document ID: 115651329
Document characters: 18616
Tokens available: 4005
Tokens used: 1024

NER classes: 13
Label mapping: {'O': 0, 'APP': 1, 'RESP': 2, 'A.COUNSEL': 3, 'R.COUNSEL': 4, 'JUDGE': 5, 'WIT': 6, 'AUTH': 7, 'COURT': 8, 'STAT': 9, 'PREC': 10, 'DATE': 11, 'CASENO': 12}

FORWARD-PASS RESULTS
Input shape:        (1, 1024)
Hidden-state shape: (1, 1024, 1024)
NER logits shape:   (1, 1024, 13)
Hidden dtype:       torch.float32
Logits dtype:       torch.float32
Hidden device:      cuda:0

PREFLIGHT STATUS: PASS
Phase-1 chec

## Arch4 Phase-2 — Forward-Pass Preflight

The initial preflight stopped because `Architecture4` was not defined
in the active Kaggle notebook session.

The original Arch4 model definitions were loaded separately from
the Phase-1 training loop to avoid retraining the language model.

The preflight must verify:

- Strict checkpoint loading with zero missing or unexpected keys.
- Input shape: `(1, 1024)`.
- Hidden-state shape: `(1, 1024, 1024)`.
- NER-logit shape: `(1, 1024, 13)`.
- Finite hidden-state and logit values.

The Arch4 model has **201,443,328 unique parameters**. Its language-model
head shares the token embedding weights, so summing every state-dictionary
tensor entry without accounting for shared weights overcounts parameters.

**Status:** Pending successful completion of the forward-pass preflight.

# Phase 2: Arch4 Backbone Forward-Pass Verification

## Objective

Verify that the Phase-1 Arch4 checkpoint can be loaded into the
original architecture and connected to a token-classification head
for Legal Named Entity Recognition (L-NER).

## Checkpoint

| Property | Value |
|---|---:|
| Training step | 10,000 |
| Best validation loss | 2.475979 |
| Vocabulary size | 40,000 |
| Model dimension | 1,024 |
| Transformer layers | 12 |
| Query heads | 16 |
| KV heads | 8 |
| Feed-forward dimension | 3,328 |
| Maximum sequence length | 1,024 |
| Local attention window | 256 |
| Global attention frequency | Every 3 layers |
| State-dict parameter entries | 242,403,328 |

The checkpoint parameter-entry count includes the token embedding matrix
of shape `(40,000, 1,024)`. The transformer/core parameter count is
201,443,328.

## NER Classification Head

The classification head maps each contextual token representation
to one of 13 classes:

- 12 entity classes from `labels.txt`
- 1 non-entity class, `O`

The classification layer maps a 1,024-dimensional hidden state to
13 output logits.

## Forward-Pass Pipeline

1. Load the Phase-1 checkpoint.
2. Instantiate the original Arch4 architecture.
3. Load the checkpoint with strict state-dictionary matching.
4. Tokenize a real Phase-2 training document.
5. Construct a 1,024-token input window.
6. Extract contextual hidden states.
7. Apply the 13-class NER classification head.
8. Validate tensor dimensions and numerical values.

## Expected Tensor Dimensions

| Tensor | Expected shape |
|---|---|
| Input token IDs | `(1, 1024)` |
| Contextual hidden states | `(1, 1024, 1024)` |
| NER logits | `(1, 1024, 13)` |

## Important Limitation

This is an integration and shape-validation test only.
The NER head is randomly initialized, and no optimization or
fine-tuning takes place in this step. The output logits must not
be treated as meaningful NER predictions.

## Status

Mark this verification as **PASS** only if:

- Checkpoint loading succeeds with strict matching.
- No missing or unexpected state-dictionary keys are reported.
- Hidden states have shape `(1, 1024, 1024)`.
- NER logits have shape `(1, 1024, 13)`.
- Neither tensor contains NaN or infinity.

In [13]:
# ============================================================
# PHASE 2 — TOKEN/OFFSET ALIGNMENT AUDIT
# ============================================================
# Runs on all 70 training documents.
# No model training.
#
# Checks:
# 1. Gold spans overlap token offsets.
# 2. Gold start/end boundaries align with token boundaries.
# 3. Tokens are not assigned to multiple gold entities.
# 4. Gold spans can be reconstructed through token labels.
#
# Tokenizer offsets are corrected by stripping leading
# whitespace from each token's start offset.
# ============================================================

import json
from pathlib import Path
from collections import Counter, defaultdict
from tokenizers import Tokenizer

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

NER_ROOT = Path("/kaggle/working/ner_data/Phase_2_LNER")

FOLD1_TEXT = NER_ROOT / "Train" / "fold1-text.json"
FOLD1_GOLD = NER_ROOT / "Train" / "fold1-gold.json"

FOLD2_TEXT = NER_ROOT / "Train" / "fold2-text.json"
FOLD2_GOLD = NER_ROOT / "Train" / "fold2-gold.json"

LABELS_FILE = NER_ROOT / "Train" / "labels.txt"

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/achiver07/"
    "phase1-tokens-v40k-clean1/tokenizer.json"
)

for path in [
    FOLD1_TEXT, FOLD1_GOLD,
    FOLD2_TEXT, FOLD2_GOLD,
    LABELS_FILE, TOKENIZER_PATH
]:
    assert path.is_file(), f"Missing file: {path}"

with open(LABELS_FILE, encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]

with open(FOLD1_TEXT, encoding="utf-8") as f:
    fold1_text = json.load(f)

with open(FOLD1_GOLD, encoding="utf-8") as f:
    fold1_gold = json.load(f)

with open(FOLD2_TEXT, encoding="utf-8") as f:
    fold2_text = json.load(f)

with open(FOLD2_GOLD, encoding="utf-8") as f:
    fold2_gold = json.load(f)

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

assert set(fold1_text) == set(fold1_gold)
assert set(fold2_text) == set(fold2_gold)
assert not (set(fold1_text) & set(fold2_text)), (
    "Duplicate document IDs found across the training folds."
)

documents = [
    (doc_id, text, fold1_gold[doc_id])
    for doc_id, text in fold1_text.items()
]

documents.extend(
    (doc_id, text, fold2_gold[doc_id])
    for doc_id, text in fold2_text.items()
)

valid_labels = set(ENTITY_LABELS)

# ------------------------------------------------------------
# 2. Correct tokenizer offsets
# ------------------------------------------------------------

def correct_offsets(text, raw_offsets):
    """
    Strip leading whitespace from token start offsets.

    Token end offsets remain unchanged so that the token's
    lexical end position is preserved.
    """
    corrected = []

    for start, end in raw_offsets:
        start = int(start)
        end = int(end)

        if not (0 <= start <= end <= len(text)):
            raise ValueError(
                f"Invalid tokenizer offset {(start, end)} "
                f"for text length {len(text)}"
            )

        while start < end and text[start].isspace():
            start += 1

        corrected.append((start, end))

    return corrected


# ------------------------------------------------------------
# 3. Statistics
# ------------------------------------------------------------

entity_count = Counter()
boundary_exact_count = Counter()
no_overlapping_token_count = Counter()
token_conflict_count = Counter()
roundtrip_exact_count = Counter()

total_gold = 0
total_tokens = 0
bad_gold_offsets = 0
bad_entity_text = 0
token_conflicts = 0
no_token_overlap = 0

all_gold_spans = Counter()
all_predicted_spans = Counter()

boundary_failure_examples = []
conflict_examples = []

# ------------------------------------------------------------
# 4. Process every document
# ------------------------------------------------------------

for doc_id, text, entities in documents:
    encoding = tokenizer.encode(text)

    offsets = correct_offsets(
        text,
        encoding.offsets
    )

    n_tokens = len(offsets)
    total_tokens += n_tokens

    # O = outside entity
    tags = ["O"] * n_tokens

    # -1 means token has not been assigned to any entity.
    owners = [-1] * n_tokens

    # Preserve the complete gold spans for strict evaluation.
    gold_spans_doc = []

    for entity_index, ent in enumerate(entities):
        if len(ent) < 3:
            raise ValueError(
                f"Invalid gold entity in document {doc_id}: {ent}"
            )

        start, end, label = int(ent[0]), int(ent[1]), ent[2]

        total_gold += 1
        entity_count[label] += 1

        if label not in valid_labels:
            raise ValueError(
                f"Unknown label {label!r} in document {doc_id}"
            )

        if not (0 <= start < end <= len(text)):
            bad_gold_offsets += 1
            continue

        if len(ent) >= 4 and text[start:end] != ent[3]:
            bad_entity_text += 1

        gold_span = (start, end, label)
        gold_spans_doc.append(gold_span)
        all_gold_spans[(doc_id, start, end, label)] += 1

        # Find tokens overlapping this gold character span.
        overlapping = [
            i
            for i, (tok_start, tok_end) in enumerate(offsets)
            if tok_start < tok_end
            and tok_end > start
            and tok_start < end
        ]

        if not overlapping:
            no_token_overlap += 1
            no_overlapping_token_count[label] += 1

            if len(boundary_failure_examples) < 10:
                boundary_failure_examples.append(
                    (doc_id, start, end, label, text[start:end])
                )

            continue

        first_token = overlapping[0]
        last_token = overlapping[-1]

        # Check if both character boundaries coincide with
        # the corrected token boundaries.
        boundaries_exact = (
            offsets[first_token][0] == start
            and offsets[last_token][1] == end
        )

        if boundaries_exact:
            boundary_exact_count[label] += 1
        elif len(boundary_failure_examples) < 10:
            boundary_failure_examples.append(
                (
                    doc_id,
                    start,
                    end,
                    label,
                    text[start:end],
                    offsets[first_token],
                    offsets[last_token]
                )
            )

        # Assign one entity label to each overlapping token.
        for token_index in overlapping:
            previous_owner = owners[token_index]

            if previous_owner != -1 and previous_owner != entity_index:
                token_conflicts += 1
                token_conflict_count[label] += 1

                if len(conflict_examples) < 10:
                    conflict_examples.append(
                        (
                            doc_id,
                            token_index,
                            offsets[token_index],
                            owners[token_index],
                            entity_index
                        )
                    )

            owners[token_index] = entity_index
            tags[token_index] = label

    # --------------------------------------------------------
    # 5. Decode token labels back into character spans
    # --------------------------------------------------------

    predicted_spans_doc = []
    i = 0

    while i < n_tokens:
        label = tags[i]
        start_offset, end_offset = offsets[i]

        if label == "O" or start_offset >= end_offset:
            i += 1
            continue

        j = i + 1

        # Merge consecutive tokens with the same entity label.
        while (
            j < n_tokens
            and tags[j] == label
            and offsets[j][0] < offsets[j][1]
        ):
            j += 1

        char_start = offsets[i][0]
        char_end = offsets[j - 1][1]

        if char_start < char_end:
            predicted_span = (
                char_start,
                char_end,
                label
            )

            predicted_spans_doc.append(predicted_span)
            all_predicted_spans[
                (doc_id, char_start, char_end, label)
            ] += 1

        i = j

    # Count exact round-trip matches by entity label.
    gold_doc_counter = Counter(gold_spans_doc)
    predicted_doc_counter = Counter(predicted_spans_doc)

    exact_matches = gold_doc_counter & predicted_doc_counter

    for (_, _, label), count in exact_matches.items():
        roundtrip_exact_count[label] += count


# ------------------------------------------------------------
# 6. Report results
# ------------------------------------------------------------

print("=" * 72)
print("PHASE-2 TOKEN/OFFSET ALIGNMENT AUDIT")
print("=" * 72)

print(f"Training documents:             {len(documents)}")
print(f"Documents in Fold 1:             {len(fold1_text)}")
print(f"Documents in Fold 2:             {len(fold2_text)}")
print(f"Total tokenized tokens:          {total_tokens:,}")
print(f"Total gold entities:             {total_gold:,}")

print("\nData integrity:")
print(f"Invalid gold offsets:            {bad_gold_offsets}")
print(f"Entity-text mismatches:          {bad_entity_text}")

print("\nToken alignment:")
print(f"Entities with no overlapping token: {no_token_overlap}")
print(f"Tokens assigned to multiple entities: {token_conflicts}")

print("\n" + "-" * 72)
print("PER-LABEL ALIGNMENT")
print("-" * 72)

print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'BOUNDARY OK':>14}"
    f"{'ROUNDTRIP OK':>15}"
    f"{'NO TOKEN':>12}"
)

for label in ENTITY_LABELS:
    print(
        f"{label:<12}"
        f"{entity_count[label]:>8}"
        f"{boundary_exact_count[label]:>14}"
        f"{roundtrip_exact_count[label]:>15}"
        f"{no_overlapping_token_count[label]:>12}"
    )

strict_tp = sum(
    (all_gold_spans & all_predicted_spans).values()
)

# Exact strict-span metrics for this tokenizer-only test.
n_predicted = sum(all_predicted_spans.values())
n_gold = sum(all_gold_spans.values())

precision = strict_tp / n_predicted if n_predicted else 0.0
recall = strict_tp / n_gold if n_gold else 0.0

print("\n" + "-" * 72)
print("STRICT SPAN ROUND-TRIP")
print("-" * 72)

print(f"Gold spans:       {n_gold:,}")
print(f"Decoded spans:    {n_predicted:,}")
print(f"Exact matches:    {strict_tp:,}")
print(f"Strict precision: {precision:.4%}")
print(f"Strict recall:    {recall:.4%}")

if boundary_failure_examples:
    print("\nFirst boundary/no-overlap examples:")
    for example in boundary_failure_examples[:10]:
        print(example)

if conflict_examples:
    print("\nFirst token-conflict examples:")
    for example in conflict_examples[:10]:
        print(example)

print("\n" + "=" * 72)
print("AUDIT COMPLETE")
print("=" * 72)

print(
    "Note: these scores measure tokenizer/label round-trip "
    "recoverability, NOT trained-model NER performance."
)

PHASE-2 TOKEN/OFFSET ALIGNMENT AUDIT
Training documents:             70
Documents in Fold 1:             35
Documents in Fold 2:             35
Total tokenized tokens:          457,384
Total gold entities:             8,393

Data integrity:
Invalid gold offsets:            0
Entity-text mismatches:          0

Token alignment:
Entities with no overlapping token: 0
Tokens assigned to multiple entities: 0

------------------------------------------------------------------------
PER-LABEL ALIGNMENT
------------------------------------------------------------------------
LABEL           GOLD   BOUNDARY OK   ROUNDTRIP OK    NO TOKEN
APP              407           400            398           0
RESP             381           374            372           0
A.COUNSEL        195           194            194           0
R.COUNSEL        184           184            184           0
JUDGE            245           245            233           0
WIT              187           187            187     

# Phase 2 — Arch4 Forward-Pass Preflight

## Objective

Verify compatibility between the Phase-1 Arch4 checkpoint and the
Phase-2 Legal Named Entity Recognition (L-NER) classification head.

## Checkpoint Verification

| Property | Result |
|---|---|
| Checkpoint | `best.pt` |
| Training step | 10,000 |
| Best validation loss | 2.475979 |
| Vocabulary size | 40,000 |
| Model dimension | 1,024 |
| Transformer layers | 12 |
| Query heads | 16 |
| KV heads | 8 |
| Feed-forward dimension | 3,328 |
| Maximum sequence length | 1,024 |
| Strict checkpoint loading | PASS |
| Missing state-dictionary keys | 0 |
| Unexpected state-dictionary keys | 0 |

## Forward-Pass Verification

A real document from Fold 1 was tokenized and used for the
forward-pass test.

| Component | Shape | Status |
|---|---|---|
| Input token IDs | `(1, 1024)` | PASS |
| Contextual hidden states | `(1, 1024, 1024)` | PASS |
| NER logits | `(1, 1024, 13)` | PASS |
| Numerical finiteness | No NaN/Inf | PASS |

The NER head contains 13 output classes: 12 entity labels and one
outside-entity class, `O`.

## Status

**Forward-pass preflight: PASS**

The Phase-1 checkpoint loads successfully, and the Arch4 backbone
produces contextual hidden states with the expected dimensions.
The NER classification head also produces valid output dimensions.

No Phase-2 training has been performed in this step.

## Next Step

Run the token-offset and character-span round-trip audit across all
70 training documents. Confirm boundary recoverability and verify
that no tokens are assigned to multiple gold entities before
starting the Phase-2 training experiments.

In [14]:
# ============================================================
# PHASE 2 — BIO TOKEN/OFFSET ROUND-TRIP AUDIT
# ============================================================
# Corrects the previous audit's entity-merging problem.
#
# Encoding:
#   O        = outside an entity
#   B-LABEL  = first token of an entity
#   I-LABEL  = subsequent tokens of that entity
#
# Total classes: 25 (12 labels, BIO, plus O)
# ============================================================

from collections import Counter

assert "documents" in globals()
assert "tokenizer" in globals()
assert "correct_offsets" in globals()
assert "ENTITY_LABELS" in globals()

BIO_TAGS = (
    ["O"]
    + [f"B-{label}" for label in ENTITY_LABELS]
    + [f"I-{label}" for label in ENTITY_LABELS]
)

TAG2ID = {tag: idx for idx, tag in enumerate(BIO_TAGS)}
ID2TAG = {idx: tag for tag, idx in TAG2ID.items()}

assert len(TAG2ID) == 25
assert len(ID2TAG) == 25

print("BIO classes:", len(TAG2ID))
print("Label mapping:", TAG2ID)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

entity_count = Counter()
boundary_exact_count = Counter()
roundtrip_exact_count = Counter()

all_gold_spans = Counter()
all_decoded_spans = Counter()

total_tokens = 0
total_gold = 0
no_token_overlap = 0
token_conflicts = 0

boundary_failure_examples = []
conflict_examples = []

# ------------------------------------------------------------
# Process the 70 documents
# ------------------------------------------------------------

for doc_id, text, entities in documents:

    encoding = tokenizer.encode(text)
    offsets = correct_offsets(text, encoding.offsets)

    n_tokens = len(offsets)
    total_tokens += n_tokens

    # Each token receives one BIO tag.
    tags = ["O"] * n_tokens

    # Tracks which entity owns each token.
    owners = [-1] * n_tokens

    gold_doc = []

    for entity_index, ent in enumerate(entities):
        start, end, label = int(ent[0]), int(ent[1]), ent[2]

        total_gold += 1
        entity_count[label] += 1

        assert label in ENTITY_LABELS
        assert 0 <= start < end <= len(text)

        # Find token offsets overlapping this character span.
        overlapping = [
            i
            for i, (tok_start, tok_end) in enumerate(offsets)
            if tok_start < tok_end
            and tok_end > start
            and tok_start < end
        ]

        if not overlapping:
            no_token_overlap += 1
            continue

        first = overlapping[0]
        last = overlapping[-1]

        # Can a token-offset decoder reproduce this exact span?
        boundaries_exact = (
            offsets[first][0] == start
            and offsets[last][1] == end
        )

        if boundaries_exact:
            boundary_exact_count[label] += 1
        elif len(boundary_failure_examples) < 10:
            boundary_failure_examples.append(
                (
                    doc_id,
                    start,
                    end,
                    label,
                    text[start:end],
                    offsets[first],
                    offsets[last],
                )
            )

        gold_span = (start, end, label)
        gold_doc.append(gold_span)

        all_gold_spans[
            (doc_id, start, end, label)
        ] += 1

        # Assign BIO tags to overlapping tokens.
        for position, token_index in enumerate(overlapping):

            previous_owner = owners[token_index]

            if (
                previous_owner != -1
                and previous_owner != entity_index
            ):
                token_conflicts += 1

                if len(conflict_examples) < 10:
                    conflict_examples.append(
                        (
                            doc_id,
                            token_index,
                            offsets[token_index],
                            previous_owner,
                            entity_index,
                        )
                    )

                continue

            owners[token_index] = entity_index

            if position == 0:
                tags[token_index] = f"B-{label}"
            else:
                tags[token_index] = f"I-{label}"

    # --------------------------------------------------------
    # Decode BIO labels into character spans
    # --------------------------------------------------------

    decoded_doc = []
    i = 0

    while i < n_tokens:
        tag = tags[i]

        if tag == "O":
            i += 1
            continue

        if tag.startswith("B-"):
            label = tag[2:]

        elif tag.startswith("I-"):
            # Robustly handle an orphan I-tag as a new span.
            label = tag[2:]

        else:
            i += 1
            continue

        char_start = offsets[i][0]
        j = i + 1

        # Continue through I-tags of this same entity type.
        while (
            j < n_tokens
            and tags[j] == f"I-{label}"
        ):
            j += 1

        char_end = offsets[j - 1][1]

        if char_start < char_end:
            span = (char_start, char_end, label)
            decoded_doc.append(span)

            all_decoded_spans[
                (doc_id, char_start, char_end, label)
            ] += 1

        i = j

    # Count exact span matches by entity type.
    gold_counter = Counter(gold_doc)
    decoded_counter = Counter(decoded_doc)

    exact_matches = gold_counter & decoded_counter

    for (_, _, label), count in exact_matches.items():
        roundtrip_exact_count[label] += count


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BIO ROUND-TRIP RESULTS")
print("=" * 78)

print(f"Documents:                 {len(documents)}")
print(f"Tokens:                    {total_tokens:,}")
print(f"Gold entities:             {total_gold:,}")
print(f"Entities without tokens:   {no_token_overlap}")
print(f"Token conflicts:           {token_conflicts}")

print("\nPer-label results:")
print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'BOUNDARY OK':>14}"
    f"{'BIO ROUNDTRIP':>16}"
    f"{'BOUNDARY FAIL':>15}"
)

for label in ENTITY_LABELS:
    gold_n = entity_count[label]
    boundary_n = boundary_exact_count[label]
    roundtrip_n = roundtrip_exact_count[label]

    print(
        f"{label:<12}"
        f"{gold_n:>8}"
        f"{boundary_n:>14}"
        f"{roundtrip_n:>16}"
        f"{gold_n - boundary_n:>15}"
    )

# Strict exact-span comparison.
true_positives = sum(
    (all_gold_spans & all_decoded_spans).values()
)

n_gold = sum(all_gold_spans.values())
n_decoded = sum(all_decoded_spans.values())

strict_precision = (
    true_positives / n_decoded
    if n_decoded else 0.0
)

strict_recall = (
    true_positives / n_gold
    if n_gold else 0.0
)

print("\nStrict span metrics:")
print(f"Gold spans:       {n_gold:,}")
print(f"Decoded spans:    {n_decoded:,}")
print(f"Exact matches:    {true_positives:,}")
print(f"Strict precision: {strict_precision:.4%}")
print(f"Strict recall:    {strict_recall:.4%}")

if boundary_failure_examples:
    print("\nExamples of token-boundary mismatches:")
    for example in boundary_failure_examples:
        print(example)

if conflict_examples:
    print("\nExamples of token conflicts:")
    for example in conflict_examples:
        print(example)

print("\nBIO audit complete.")
print("These metrics evaluate alignment, not trained-model performance.")

BIO classes: 25
Label mapping: {'O': 0, 'B-APP': 1, 'B-RESP': 2, 'B-A.COUNSEL': 3, 'B-R.COUNSEL': 4, 'B-JUDGE': 5, 'B-WIT': 6, 'B-AUTH': 7, 'B-COURT': 8, 'B-STAT': 9, 'B-PREC': 10, 'B-DATE': 11, 'B-CASENO': 12, 'I-APP': 13, 'I-RESP': 14, 'I-A.COUNSEL': 15, 'I-R.COUNSEL': 16, 'I-JUDGE': 17, 'I-WIT': 18, 'I-AUTH': 19, 'I-COURT': 20, 'I-STAT': 21, 'I-PREC': 22, 'I-DATE': 23, 'I-CASENO': 24}

BIO ROUND-TRIP RESULTS
Documents:                 70
Tokens:                    457,384
Gold entities:             8,393
Entities without tokens:   0
Token conflicts:           0

Per-label results:
LABEL           GOLD   BOUNDARY OK   BIO ROUNDTRIP  BOUNDARY FAIL
APP              407           400             400              7
RESP             381           374             374              7
A.COUNSEL        195           194             194              1
R.COUNSEL        184           184             184              0
JUDGE            245           245             245              0
WIT          

# Phase 2 — Token Alignment Audit

## Initial Audit Results

The tokenizer and gold annotations were checked across 70 training
documents containing 8,393 entities and approximately 457,000 tokens.

| Check | Result |
|---|---:|
| Invalid gold offsets | 0 |
| Entity-text mismatches | 0 |
| Entities without an overlapping token | 0 |
| Token conflicts | 0 |
| Gold spans aligned to token boundaries | 8,240 / 8,393 |
| Exact matches using flat entity labels | 8,069 / 8,393 |
| Initial strict round-trip precision | 97.46% |
| Initial strict round-trip recall | 96.14% |

## Identified Limitation

The initial audit used a flat label scheme with 12 entity labels and
one outside-entity label (`O`).

Its decoder merged consecutive tokens with identical entity labels.
Consequently, two consecutive entities of the same type could be
incorrectly decoded as one entity.

The discrepancy was especially pronounced for `CASENO`:

- Boundary-aligned entities: 759
- Exact round-trip matches: 621

## Corrective Action

Evaluate BIO tagging, using:

- `O`: outside an entity
- `B-LABEL`: first token of an entity
- `I-LABEL`: subsequent tokens of the same entity

This produces 25 classes for the 12 entity types.

BIO tagging preserves boundaries between consecutive entities with
the same label. It does not resolve entity boundaries that fall
inside tokenizer tokens.

## Interpretation

The initial round-trip score measures the combined effects of token
boundary alignment and the flat-label decoder. It is not a measurement
of trained-model performance or the true recoverability ceiling.

The BIO round-trip audit must be run before the training configuration
is finalized.

In [15]:
# ============================================================
# PHASE 2 — DOCUMENT-LEVEL TRAIN/VALIDATION SPLIT
# ============================================================
# Requires the variables created by the previous audit:
#   documents
#   ENTITY_LABELS
#
# All windows from a document must remain in the same partition.
# ============================================================

import json
import random
from pathlib import Path
from collections import Counter

NER_ROOT = Path("/kaggle/working/ner_data/Phase_2_LNER")
FOLD3_TEXT = NER_ROOT / "Test" / "fold3-text.json"

# ------------------------------------------------------------
# 1. Organize the 70 training documents
# ------------------------------------------------------------

doc_map = {
    doc_id: {
        "text": text,
        "entities": entities,
    }
    for doc_id, text, entities in documents
}

doc_ids = list(doc_map)

assert len(doc_map) == 70, (
    f"Expected 70 unique training documents; found {len(doc_map)}."
)

with open(FOLD3_TEXT, "r", encoding="utf-8") as f:
    test_texts = json.load(f)

test_ids = set(test_texts.keys())
training_ids = set(doc_ids)

assert training_ids.isdisjoint(test_ids), (
    "Training and test document IDs overlap."
)

# Count gold entities per label in each document.
doc_label_counts = {}

for doc_id, item in doc_map.items():
    doc_label_counts[doc_id] = Counter(
        entity[2]
        for entity in item["entities"]
    )

total_label_counts = Counter()

for counts in doc_label_counts.values():
    total_label_counts.update(counts)

assert set(total_label_counts) == set(ENTITY_LABELS)

# ------------------------------------------------------------
# 2. Search for a reproducible 20% document holdout
# ------------------------------------------------------------

VAL_FRACTION = 0.20
N_VAL = round(len(doc_ids) * VAL_FRACTION)

assert N_VAL == 14

best_split = None
best_score = float("inf")

# Find a split where every label occurs in both partitions.
for trial in range(5000):
    shuffled = doc_ids.copy()

    random.Random(20261009 + trial).shuffle(shuffled)

    candidate_val = set(shuffled[:N_VAL])
    candidate_train = set(shuffled[N_VAL:])

    val_counts = Counter()

    for doc_id in candidate_val:
        val_counts.update(doc_label_counts[doc_id])

    train_counts = Counter()

    for doc_id in candidate_train:
        train_counts.update(doc_label_counts[doc_id])

    # Require every entity type to occur in both partitions.
    if any(val_counts[label] == 0 for label in ENTITY_LABELS):
        continue

    if any(train_counts[label] == 0 for label in ENTITY_LABELS):
        continue

    # Prefer a validation set whose per-label entity proportions
    # are reasonably close to the 20% document holdout fraction.
    score = sum(
        abs(
            val_counts[label] / total_label_counts[label]
            - VAL_FRACTION
        )
        for label in ENTITY_LABELS
    )

    if score < best_score:
        best_score = score

        best_split = (
            candidate_train.copy(),
            candidate_val.copy(),
            train_counts.copy(),
            val_counts.copy(),
            trial,
        )

assert best_split is not None, (
    "No valid split found after 5000 attempts. "
    "Inspect the document-level label distribution."
)

TRAIN_DOC_IDS, VAL_DOC_IDS, train_counts, val_counts, split_trial = (
    best_split
)

# ------------------------------------------------------------
# 3. Materialize document partitions
# ------------------------------------------------------------

train_docs = {
    doc_id: doc_map[doc_id]
    for doc_id in TRAIN_DOC_IDS
}

val_docs = {
    doc_id: doc_map[doc_id]
    for doc_id in VAL_DOC_IDS
}

assert len(train_docs) == 56
assert len(val_docs) == 14

assert set(train_docs).isdisjoint(set(val_docs))
assert set(train_docs) | set(val_docs) == training_ids

assert set(train_docs).isdisjoint(test_ids)
assert set(val_docs).isdisjoint(test_ids)

# ------------------------------------------------------------
# 4. Report partition statistics
# ------------------------------------------------------------

print("=" * 72)
print("DOCUMENT-LEVEL SPLIT")
print("=" * 72)

print("Unique training documents:", len(doc_map))
print("Training documents:", len(train_docs))
print("Validation documents:", len(val_docs))
print("Official test documents:", len(test_texts))
print("Selected split trial:", split_trial)

print("\nEntity counts:")
print(f"{'LABEL':<12}{'TRAIN':>10}{'VALIDATION':>14}{'TOTAL':>10}")

for label in ENTITY_LABELS:
    print(
        f"{label:<12}"
        f"{train_counts[label]:>10}"
        f"{val_counts[label]:>14}"
        f"{total_label_counts[label]:>10}"
    )

print("\nPartition checks:")
print("Train/validation overlap:", len(set(train_docs) & set(val_docs)))
print("Train/test overlap:", len(set(train_docs) & test_ids))
print("Validation/test overlap:", len(set(val_docs) & test_ids))

print("\nSTATUS: PASS")
print("All documents belong to exactly one training/validation partition.")
print("The official test fold remains untouched.")

DOCUMENT-LEVEL SPLIT
Unique training documents: 70
Training documents: 56
Validation documents: 14
Official test documents: 35
Selected split trial: 4099

Entity counts:
LABEL            TRAIN    VALIDATION     TOTAL
APP                316            91       407
RESP               307            74       381
A.COUNSEL          160            35       195
R.COUNSEL          143            41       184
JUDGE              195            50       245
WIT                147            40       187
AUTH               827           176      1003
COURT              829           184      1013
STAT              1065           195      1260
PREC               950           233      1183
DATE              1261           314      1575
CASENO             608           152       760

Partition checks:
Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0

STATUS: PASS
All documents belong to exactly one training/validation partition.
The official test fold remains untouched.


In [16]:
# ============================================================
# PHASE 2 — BIO LABELS + WINDOWED DATASET PREPARATION
# ============================================================
#
# Input:
#   56 training documents
#   14 validation documents
#   35 untouched test documents
#
# Configuration:
#   Max sequence length: 1024
#   Stride:               768
#   Overlap:              256
#   Number of BIO classes: 25
#
# Guarantees:
#   - Documents do not cross dataset partitions.
#   - Each real token is supervised in exactly one window.
#   - Overlapping tokens remain available as context.
#   - Padding and non-character special tokens use label -100.
#   - The original Phase-1 checkpoint is not modified.
# ============================================================

import json
import torch
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MAX_LEN = 1024
STRIDE = 768
OVERLAP = MAX_LEN - STRIDE
PAD_ID = 0

OUT_DIR = Path("/kaggle/working/phase2_arch4_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert MAX_LEN % 256 == 0
assert STRIDE % 256 == 0
assert OVERLAP == 256

assert "train_docs" in globals()
assert "val_docs" in globals()
assert "test_texts" in globals()
assert "tokenizer" in globals()
assert "correct_offsets" in globals()
assert "ENTITY_LABELS" in globals()

assert len(train_docs) == 56
assert len(val_docs) == 14
assert len(test_texts) == 35

# Confirm that each document belongs to only one partition.
TRAIN_IDS = set(train_docs)
VAL_IDS = set(val_docs)
TEST_IDS = set(test_texts)

assert TRAIN_IDS.isdisjoint(VAL_IDS)
assert TRAIN_IDS.isdisjoint(TEST_IDS)
assert VAL_IDS.isdisjoint(TEST_IDS)
assert TRAIN_IDS | VAL_IDS == set(documents[i][0] for i in range(len(documents)))

# ------------------------------------------------------------
# 2. Define the 25 BIO classes
# ------------------------------------------------------------

BIO_TAGS = (
    ["O"]
    + [f"B-{label}" for label in ENTITY_LABELS]
    + [f"I-{label}" for label in ENTITY_LABELS]
)

TAG2ID = {
    tag: idx
    for idx, tag in enumerate(BIO_TAGS)
}

ID2TAG = {
    idx: tag
    for tag, idx in TAG2ID.items()
}

NUM_CLASSES = len(TAG2ID)

assert NUM_CLASSES == 25
assert TAG2ID["O"] == 0

print("BIO classes:", NUM_CLASSES)
print("Window length:", MAX_LEN)
print("Stride:", STRIDE)
print("Overlap:", OVERLAP)

# ------------------------------------------------------------
# 3. Convert a document's character spans into BIO token labels
# ------------------------------------------------------------

def tokenize_and_label(doc_id, item):
    """
    Tokenize one complete document and assign BIO tags.

    Gold spans use exclusive end offsets:
        [start_character, end_character, label, entity_text]

    If a character boundary falls inside a token, the entity is
    assigned to the token(s) overlapping the character span.
    Exact character-boundary limitations remain documented.
    """

    text = item["text"]
    entities = item["entities"]

    encoding = tokenizer.encode(text)

    token_ids = list(encoding.ids)

    offsets = correct_offsets(
        text,
        encoding.offsets
    )

    assert len(token_ids) == len(offsets)
    assert len(token_ids) > 0

    assert all(
        0 <= token_id < 40000
        for token_id in token_ids
    ), "Token ID is outside the Phase-1 vocabulary."

    n_tokens = len(token_ids)

    # Initial label for every token is O.
    token_tags = ["O"] * n_tokens

    # Track entity ownership so conflicts cannot pass silently.
    token_owner = [-1] * n_tokens

    for entity_index, entity in enumerate(entities):

        start = int(entity[0])
        end = int(entity[1])
        label = entity[2]

        assert label in ENTITY_LABELS, (
            f"Unknown entity label: {label}"
        )

        assert 0 <= start < end <= len(text), (
            f"Invalid span in document {doc_id}: {entity}"
        )

        # Find every non-empty token overlapping this entity.
        overlapping = [
            token_index
            for token_index, (tok_start, tok_end) in enumerate(offsets)
            if (
                tok_start < tok_end
                and tok_end > start
                and tok_start < end
            )
        ]

        if not overlapping:
            raise ValueError(
                f"Entity has no overlapping token: "
                f"doc={doc_id}, entity={entity}"
            )

        # Fail if two distinct entities compete for the same token.
        for token_index in overlapping:

            if token_owner[token_index] not in (-1, entity_index):
                raise ValueError(
                    "Token/entity conflict: "
                    f"doc={doc_id}, "
                    f"token={token_index}, "
                    f"offset={offsets[token_index]}, "
                    f"entity={entity}"
                )

            token_owner[token_index] = entity_index

        # BIO assignments.
        token_tags[overlapping[0]] = f"B-{label}"

        for token_index in overlapping[1:]:
            token_tags[token_index] = f"I-{label}"

    # Convert labels into integer IDs.
    tag_ids = [TAG2ID[tag] for tag in token_tags]

    return {
        "text": text,
        "token_ids": token_ids,
        "offsets": offsets,
        "tag_ids": tag_ids,
        "entities": entities,
    }


# ------------------------------------------------------------
# 4. Create windows with unique supervision ownership
# ------------------------------------------------------------

def create_windows(doc_id, encoded_doc):
    """
    Construct fixed-size windows.

    Window 0 supervises global token positions [0, 1024).

    Each subsequent window starts 768 tokens later and supervises
    its local positions [256, 1024).

    Consequently, the supervised global ranges are contiguous:
        [0, 1024)
        [1024, 1792)
        [1792, 2560)
        ...

    The first 256 tokens of later windows remain available as
    context but receive label -100 in that window.
    """

    ids = encoded_doc["token_ids"]
    offsets = encoded_doc["offsets"]
    tag_ids = encoded_doc["tag_ids"]

    n_tokens = len(ids)

    # Always include the first window.
    starts = [0]

    # Add more windows only while the existing window does not
    # cover the end of the document.
    while starts[-1] + MAX_LEN < n_tokens:
        starts.append(starts[-1] + STRIDE)

    windows = []

    # Every original token must belong to exactly one supervised
    # ownership region, including special tokens that will be
    # masked from the loss.
    owner_counts = [0] * n_tokens

    for window_index, start in enumerate(starts):

        end = min(start + MAX_LEN, n_tokens)
        valid_length = end - start

        ids_slice = ids[start:end]
        assert len(ids_slice) == valid_length

        # Right-pad the input to 1024 tokens.
        input_ids = (
            ids_slice
            + [PAD_ID] * (MAX_LEN - valid_length)
        )

        # Ignore padding and context-only overlap in the loss.
        labels = [-100] * MAX_LEN

        owner_local_start = (
            0 if window_index == 0 else OVERLAP
        )

        # Only the owned region contributes to the loss.
        for local_index in range(
            owner_local_start,
            valid_length
        ):

            global_index = start + local_index

            owner_counts[global_index] += 1

            tok_start, tok_end = offsets[global_index]

            # Zero-width special tokens are not training targets.
            if tok_start < tok_end:
                labels[local_index] = tag_ids[global_index]

        owner_global_start = start + owner_local_start

        windows.append({
            "input_ids": input_ids,
            "labels": labels,
            "metadata": {
                "doc_id": str(doc_id),
                "window_start": start,
                "valid_length": valid_length,
                "token_count": n_tokens,
                "owner_local_start": owner_local_start,
                "owner_global_start": owner_global_start,
                "owner_global_end": end,
            },
        })

    # Every token must belong to exactly one window ownership range.
    if not all(count == 1 for count in owner_counts):
        bad_positions = [
            (i, count)
            for i, count in enumerate(owner_counts)
            if count != 1
        ][:10]

        raise AssertionError(
            f"Ownership coverage failure for {doc_id}: "
            f"{bad_positions}"
        )

    expected_supervised = sum(
        1
        for tok_start, tok_end in offsets
        if tok_start < tok_end
    )

    actual_supervised = sum(
        label != -100
        for window in windows
        for label in window["labels"]
    )

    assert actual_supervised == expected_supervised, (
        f"Supervision mismatch for {doc_id}: "
        f"expected {expected_supervised}, "
        f"found {actual_supervised}"
    )

    return windows, expected_supervised


# ------------------------------------------------------------
# 5. Build a training or validation partition
# ------------------------------------------------------------

def build_partition(partition_docs, partition_name):

    all_input_ids = []
    all_labels = []
    metadata = []

    label_counts = Counter()

    total_tokens = 0
    total_supervised_tokens = 0
    total_entities = 0

    for doc_id in sorted(partition_docs, key=str):

        item = partition_docs[doc_id]

        encoded_doc = tokenize_and_label(
            doc_id,
            item
        )

        doc_windows, supervised_count = create_windows(
            doc_id,
            encoded_doc
        )

        total_tokens += len(encoded_doc["token_ids"])
        total_supervised_tokens += supervised_count
        total_entities += len(item["entities"])

        for window in doc_windows:

            x = window["input_ids"]
            y = window["labels"]

            assert len(x) == MAX_LEN
            assert len(y) == MAX_LEN

            all_input_ids.append(x)
            all_labels.append(y)
            metadata.append(window["metadata"])

            # Count only supervised targets.
            for label_id in y:
                if label_id != -100:
                    label_counts[ID2TAG[label_id]] += 1

    assert len(all_input_ids) == len(all_labels)
    assert len(all_input_ids) == len(metadata)
    assert len(all_input_ids) > 0

    # Convert to compact tensors for subsequent model training.
    input_tensor = torch.tensor(
        all_input_ids,
        dtype=torch.long
    )

    label_tensor = torch.tensor(
        all_labels,
        dtype=torch.long
    )

    assert input_tensor.shape == (
        len(metadata),
        MAX_LEN
    )

    assert label_tensor.shape == (
        len(metadata),
        MAX_LEN
    )

    actual_supervised = int(
        (label_tensor != -100).sum().item()
    )

    assert actual_supervised == total_supervised_tokens

    dataset = {
        "input_ids": input_tensor,
        "labels": label_tensor,
        "metadata": metadata,
        "num_documents": len(partition_docs),
        "num_windows": len(metadata),
        "num_tokens": total_tokens,
        "num_entities": total_entities,
        "num_supervised_tokens": actual_supervised,
        "label_counts": dict(label_counts),
    }

    return dataset


# ------------------------------------------------------------
# 6. Build both partitions
# ------------------------------------------------------------

print("\nBuilding training windows...")

train_dataset = build_partition(
    train_docs,
    "train"
)

print("Building validation windows...")

val_dataset = build_partition(
    val_docs,
    "validation"
)

# ------------------------------------------------------------
# 7. Persist datasets and split manifest
# ------------------------------------------------------------

train_path = OUT_DIR / "train_windows.pt"
val_path = OUT_DIR / "validation_windows.pt"
manifest_path = OUT_DIR / "split_manifest.json"

torch.save(train_dataset, train_path)
torch.save(val_dataset, val_path)

manifest = {
    "seed_note": (
        "Uses the document-level split selected in the preceding cell."
    ),
    "max_length": MAX_LEN,
    "stride": STRIDE,
    "overlap": OVERLAP,
    "num_classes": NUM_CLASSES,
    "bio_tags": BIO_TAGS,
    "train_doc_ids": sorted(map(str, TRAIN_IDS)),
    "validation_doc_ids": sorted(map(str, VAL_IDS)),
    "test_doc_ids": sorted(map(str, TEST_IDS)),
    "train_windows": len(train_dataset["metadata"]),
    "validation_windows": len(val_dataset["metadata"]),
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PHASE-2 WINDOW DATASET SUMMARY")
print("=" * 72)

for name, dataset, path in [
    ("TRAIN", train_dataset, train_path),
    ("VALIDATION", val_dataset, val_path),
]:
    print(f"\n{name}")
    print("-" * 45)
    print("Documents:        ", dataset["num_documents"])
    print("Windows:          ", dataset["num_windows"])
    print("Token positions:  ", f"{dataset['num_tokens']:,}")
    print("Gold entities:    ", f"{dataset['num_entities']:,}")
    print(
        "Supervised tokens:",
        f"{dataset['num_supervised_tokens']:,}"
    )
    print("Saved to:         ", path)

print("\nBIO classes:", NUM_CLASSES)
print("Maximum length:", MAX_LEN)
print("Stride:", STRIDE)
print("Overlap:", OVERLAP)

print("\nPartition isolation:")
print("Train/validation overlap:", len(TRAIN_IDS & VAL_IDS))
print("Train/test overlap:", len(TRAIN_IDS & TEST_IDS))
print("Validation/test overlap:", len(VAL_IDS & TEST_IDS))

print("\nSaved manifest:", manifest_path)

print("\nSTATUS: PASS")
print("Window coverage, label dimensions, and unique supervision verified.")
print("No model training was performed.")

BIO classes: 25
Window length: 1024
Stride: 768
Overlap: 256

Building training windows...
Building validation windows...

PHASE-2 WINDOW DATASET SUMMARY

TRAIN
---------------------------------------------
Documents:         56
Windows:           502
Token positions:   377,218
Gold entities:     6,808
Supervised tokens: 377,187
Saved to:          /kaggle/working/phase2_arch4_data/train_windows.pt

VALIDATION
---------------------------------------------
Documents:         14
Windows:           105
Token positions:   80,166
Gold entities:     1,585
Supervised tokens: 80,163
Saved to:          /kaggle/working/phase2_arch4_data/validation_windows.pt

BIO classes: 25
Maximum length: 1024
Stride: 768
Overlap: 256

Partition isolation:
Train/validation overlap: 0
Train/test overlap: 0
Validation/test overlap: 0

Saved manifest: /kaggle/working/phase2_arch4_data/split_manifest.json

STATUS: PASS
Window coverage, label dimensions, and unique supervision verified.
No model training was perform

# Phase 2 — BIO Labels and Windowed Dataset

## Document-Level Split

| Partition | Documents |
|---|---:|
| Training | 56 |
| Validation | 14 |
| Official test | 35 |
| Total available documents | 105 |

Training and validation windows are generated only after the
document-level split. The official test fold remains untouched.

## BIO Classification

The token classifier uses 25 classes:

- One outside-entity class: `O`
- Twelve beginning classes: `B-LABEL`
- Twelve inside classes: `I-LABEL`

## Window Configuration

| Parameter | Value |
|---|---:|
| Maximum sequence length | 1,024 |
| Window stride | 768 |
| Overlap | 256 |
| Padding label | `-100` |
| Padding token ID | `0` |

## Unique Supervision

Overlapping tokens are retained as context. Each token position
belongs to exactly one supervision region, preventing duplicate
loss contributions from overlap.

For the first window, the full valid window is supervised.
For subsequent windows, the first 256 tokens are context-only;
supervision starts at local position 256.

Padding and zero-width special tokens are ignored by the loss.

## Data Integrity Checks

The preparation code verifies:

- Token IDs are within the Phase-1 vocabulary.
- Gold character spans have valid offsets and labels.
- No two gold entities claim the same token.
- Every document token is covered by exactly one ownership region.
- Input and label tensors have shape `(num_windows, 1024)`.
- The number of supervised labels matches the number of valid,
  non-empty token offsets.
- Training, validation, and test document IDs are disjoint.

## Output Artifacts

- `train_windows.pt`
- `validation_windows.pt`
- `split_manifest.json`

These files contain preprocessing outputs only. No Phase-2
optimization or fine-tuning is performed in this step.

In [17]:
# ============================================================
# PHASE 2 — DATASET RELOAD AND BIO CLASSIFICATION SMOKE TEST
# ============================================================
# This cell:
#   1. Reloads the saved train/validation datasets.
#   2. Validates tensor shapes, token IDs, and label IDs.
#   3. Creates the correct 25-class BIO head.
#   4. Runs one real window through the Arch4 backbone.
#   5. Computes masked BIO cross-entropy.
#
# No optimization or model training is performed.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DATA_DIR = Path("/kaggle/working/phase2_arch4_data")

TRAIN_PATH = DATA_DIR / "train_windows.pt"
VAL_PATH = DATA_DIR / "validation_windows.pt"
MANIFEST_PATH = DATA_DIR / "split_manifest.json"

MAX_LEN = 1024
VOCAB_SIZE = 40000
IGNORE_INDEX = -100
NUM_CLASSES = 25

for path in [TRAIN_PATH, VAL_PATH, MANIFEST_PATH]:
    assert path.is_file(), f"Missing artifact: {path}"

assert "backbone" in globals(), (
    "Arch4 backbone is missing. Run the checkpoint preflight first."
)

assert "TAG2ID" in globals(), (
    "BIO mapping is missing. Run the BIO-label preparation cell first."
)

assert len(TAG2ID) == NUM_CLASSES

# ------------------------------------------------------------
# 2. Reload the saved datasets
# ------------------------------------------------------------

train_data = torch.load(
    TRAIN_PATH,
    map_location="cpu",
    weights_only=False
)

val_data = torch.load(
    VAL_PATH,
    map_location="cpu",
    weights_only=False
)

print("=" * 68)
print("PHASE-2 DATASET RELOAD CHECK")
print("=" * 68)

for split_name, data in [
    ("TRAIN", train_data),
    ("VALIDATION", val_data),
]:
    x = data["input_ids"]
    y = data["labels"]
    metadata = data["metadata"]

    print(f"\n{split_name}")
    print("Documents:", data["num_documents"])
    print("Windows:", data["num_windows"])
    print("Input shape:", tuple(x.shape))
    print("Label shape:", tuple(y.shape))
    print("Supervised tokens:", int((y != IGNORE_INDEX).sum()))

    assert x.ndim == 2
    assert y.ndim == 2
    assert x.shape == y.shape
    assert x.shape[1] == MAX_LEN
    assert len(metadata) == x.shape[0]

    assert x.dtype == torch.long
    assert y.dtype == torch.long

    # All input IDs must belong to the 40k-token vocabulary.
    assert x.min().item() >= 0
    assert x.max().item() < VOCAB_SIZE

    # Valid BIO IDs are 0..24; -100 is reserved for ignored positions.
    valid_label_mask = y != IGNORE_INDEX

    assert valid_label_mask.any().item(), (
        f"{split_name} has no supervised tokens."
    )

    valid_labels = y[valid_label_mask]

    assert valid_labels.min().item() >= 0
    assert valid_labels.max().item() < NUM_CLASSES

    # Every window must contain at least one supervised token.
    supervised_per_window = valid_label_mask.sum(dim=1)

    assert (supervised_per_window > 0).all().item()

    print("Token IDs: PASS")
    print("BIO label IDs: PASS")
    print("Supervision mask: PASS")

# ------------------------------------------------------------
# 3. Create the BIO classifier
# ------------------------------------------------------------
# The previous preflight tested a 13-class head.
# This task uses 25 BIO classes, so create a fresh head.
# The Phase-1 backbone weights remain unchanged.

device = next(backbone.parameters()).device

bio_head = nn.Linear(
    backbone.cfg["D_MODEL"],
    NUM_CLASSES
).to(device)

backbone.eval()
bio_head.eval()

print("\nDevice:", device)
print("BIO classes:", NUM_CLASSES)

# ------------------------------------------------------------
# 4. Choose a real training window
# ------------------------------------------------------------

train_x = train_data["input_ids"]
train_y = train_data["labels"]

# Pick a window containing many supervised tokens.
window_index = int(
    train_y.ne(IGNORE_INDEX).sum(dim=1).argmax().item()
)

input_ids = train_x[window_index:window_index + 1].to(device)
labels = train_y[window_index:window_index + 1].to(device)

assert input_ids.shape == (1, MAX_LEN)
assert labels.shape == (1, MAX_LEN)

print("\nSelected training window:", window_index)
print("Input shape:", tuple(input_ids.shape))
print("Label shape:", tuple(labels.shape))
print("Supervised positions:", int((labels != IGNORE_INDEX).sum()))

# ------------------------------------------------------------
# 5. Forward pass and masked loss
# ------------------------------------------------------------

with torch.inference_mode():
    hidden_states = backbone.hidden_states(input_ids)

    logits = bio_head(hidden_states)

    loss = F.cross_entropy(
        logits.reshape(-1, NUM_CLASSES).float(),
        labels.reshape(-1),
        ignore_index=IGNORE_INDEX
    )

# ------------------------------------------------------------
# 6. Validate results
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("BIO CLASSIFICATION RESULTS")
print("=" * 68)

print("Hidden-state shape:", tuple(hidden_states.shape))
print("Logits shape:", tuple(logits.shape))
print("Masked BIO loss:", float(loss))
print("Hidden-state finite:", bool(torch.isfinite(hidden_states).all()))
print("Logits finite:", bool(torch.isfinite(logits).all()))
print("Loss finite:", bool(torch.isfinite(loss)))

assert hidden_states.shape == (1, MAX_LEN, 1024)
assert logits.shape == (1, MAX_LEN, NUM_CLASSES)

assert torch.isfinite(hidden_states).all().item()
assert torch.isfinite(logits).all().item()
assert torch.isfinite(loss).item()

print("\n" + "=" * 68)
print("BIO INTEGRATION SMOKE TEST: PASS")
print("=" * 68)

print("Saved datasets: loaded successfully")
print("Arch4 backbone: compatible")
print("BIO classifier: compatible")
print("Masked loss: finite")
print("Training performed: NO")

PHASE-2 DATASET RELOAD CHECK

TRAIN
Documents: 56
Windows: 502
Input shape: (502, 1024)
Label shape: (502, 1024)
Supervised tokens: 377187
Token IDs: PASS
BIO label IDs: PASS
Supervision mask: PASS

VALIDATION
Documents: 14
Windows: 105
Input shape: (105, 1024)
Label shape: (105, 1024)
Supervised tokens: 80163
Token IDs: PASS
BIO label IDs: PASS
Supervision mask: PASS

Device: cuda:0
BIO classes: 25

Selected training window: 0
Input shape: (1, 1024)
Label shape: (1, 1024)
Supervised positions: 1024

BIO CLASSIFICATION RESULTS
Hidden-state shape: (1, 1024, 1024)
Logits shape: (1, 1024, 25)
Masked BIO loss: 3.362351417541504
Hidden-state finite: True
Logits finite: True
Loss finite: True

BIO INTEGRATION SMOKE TEST: PASS
Saved datasets: loaded successfully
Arch4 backbone: compatible
BIO classifier: compatible
Masked loss: finite
Training performed: NO


# Phase 2 — Window Dataset and BIO Integration

## Dataset Preparation

The 70 labeled documents were divided at the document level before
tokenization windows were generated.

| Partition | Documents | Windows |
|---|---:|---:|
| Training | 56 | 502 |
| Validation | 14 | 105 |
| Official test | 35 | Not generated |

Training and validation contain disjoint document IDs. The official
test documents remain separate from both partitions.

## Window Configuration

| Parameter | Value |
|---|---:|
| Maximum sequence length | 1,024 |
| Stride | 768 |
| Overlap | 256 |
| Classification classes | 25 |
| Ignored label ID | -100 |

Overlap tokens provide context but do not contribute duplicate loss
terms. Padding positions and zero-width special tokens are excluded
from supervised loss.

## Saved Artifacts

- `train_windows.pt`
- `validation_windows.pt`
- `split_manifest.json`

Location:

`/kaggle/working/phase2_arch4_data/`

## BIO Integration Test

The integration test reloads the serialized tensors, validates
token and label ID ranges, and passes a real training window through
the Phase-1 Arch4 backbone and a 25-class BIO classifier.

Expected output dimensions:

- Input IDs: `(1, 1024)`
- Hidden states: `(1, 1024, 1024)`
- BIO logits: `(1, 1024, 25)`

The masked cross-entropy loss must be finite.

This is an integration test only. It does not perform optimization,
update checkpoint weights, or establish NER performance.

## Status

The dataset preparation stage has passed. The BIO integration stage
should be marked **PASS** only after the forward-pass and loss checks
complete successfully.

In [18]:
# ============================================================
# PHASE 2 — DOCUMENT-LEVEL STRICT SPAN EVALUATOR
# ============================================================
# Evaluates character spans against gold annotations.
#
# Metrics:
#   - Strict Macro-F1 (primary)
#   - Strict Macro-Precision
#   - Strict Macro-Recall
#   - Per-label precision, recall and F1
#
# No optimization or model training takes place.
# ============================================================

import torch
from collections import Counter
from contextlib import nullcontext

# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

assert "backbone" in globals()
assert "bio_head" in globals()
assert "val_data" in globals()
assert "val_docs" in globals()
assert "tokenizer" in globals()
assert "correct_offsets" in globals()
assert "ENTITY_LABELS" in globals()
assert "ID2TAG" in globals()

device = next(backbone.parameters()).device

assert len(ENTITY_LABELS) == 12
assert len(ID2TAG) == 25

backbone.eval()
bio_head.eval()

# Preserve each document's corrected character offsets and
# allocate one prediction slot per original token.
doc_ids = {
    str(doc_id): doc_id
    for doc_id in val_docs
}

doc_offsets = {}
predicted_token_tags = {}
ownership_seen = {}

for doc_id, item in val_docs.items():
    doc_id = str(doc_id)

    encoding = tokenizer.encode(item["text"])
    offsets = correct_offsets(
        item["text"],
        encoding.offsets
    )

    expected_length = len(encoding.ids)

    doc_offsets[doc_id] = offsets
    predicted_token_tags[doc_id] = [0] * expected_length
    ownership_seen[doc_id] = [False] * expected_length

# ------------------------------------------------------------
# 2. Predict each validation window
# ------------------------------------------------------------

num_windows = len(val_data["metadata"])

with torch.inference_mode():

    for window_index in range(num_windows):

        metadata = val_data["metadata"][window_index]
        doc_id = str(metadata["doc_id"])

        assert doc_id in doc_offsets, (
            f"Unknown validation document ID: {doc_id}"
        )

        input_ids = (
            val_data["input_ids"][window_index:window_index + 1]
            .to(device)
        )

        # Mixed precision for inference on CUDA.
        if device.type == "cuda":
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                hidden = backbone.hidden_states(input_ids)
                logits = bio_head(hidden)
        else:
            hidden = backbone.hidden_states(input_ids)
            logits = bio_head(hidden)

        predicted_ids = (
            logits.float()
            .argmax(dim=-1)[0]
            .cpu()
            .tolist()
        )

        window_start = int(metadata["window_start"])
        valid_length = int(metadata["valid_length"])
        owner_local_start = int(metadata["owner_local_start"])

        # Only use predictions from the region this window owns.
        # Context-only overlapping tokens are assigned by another
        # window, avoiding duplicate predictions.
        for local_idx in range(
            owner_local_start,
            valid_length
        ):

            global_idx = window_start + local_idx

            assert global_idx < len(
                predicted_token_tags[doc_id]
            )

            assert not ownership_seen[doc_id][global_idx], (
                f"Duplicate prediction ownership: "
                f"document={doc_id}, token={global_idx}"
            )

            predicted_token_tags[doc_id][global_idx] = (
                predicted_ids[local_idx]
            )

            ownership_seen[doc_id][global_idx] = True

# ------------------------------------------------------------
# 3. Verify token coverage before decoding
# ------------------------------------------------------------

for doc_id, seen in ownership_seen.items():

    missing = sum(not x for x in seen)

    assert missing == 0, (
        f"Missing predictions for document {doc_id}: "
        f"{missing} tokens"
    )

    assert len(seen) == len(doc_offsets[doc_id])

print("Validation windows evaluated:", num_windows)
print("Validation documents:", len(val_docs))
print("Token ownership coverage: PASS")

# ------------------------------------------------------------
# 4. Decode BIO labels to character spans
# ------------------------------------------------------------

def decode_bio_spans(tag_ids, offsets, doc_id):

    assert len(tag_ids) == len(offsets)

    # Zero-width tokens cannot produce character spans.
    tags = []

    for tag_id, (start, end) in zip(tag_ids, offsets):

        if start >= end:
            tags.append("O")
        else:
            tags.append(ID2TAG[int(tag_id)])

    spans = []
    i = 0

    while i < len(tags):

        tag = tags[i]

        if tag == "O":
            i += 1
            continue

        if tag.startswith("B-"):
            label = tag[2:]

        elif tag.startswith("I-"):
            # Recover gracefully if the model emits an I-tag
            # without a preceding B-tag of the same type.
            label = tag[2:]

        else:
            i += 1
            continue

        char_start = offsets[i][0]
        j = i + 1

        while (
            j < len(tags)
            and tags[j] == f"I-{label}"
        ):
            j += 1

        char_end = offsets[j - 1][1]

        if char_start < char_end:
            spans.append(
                (doc_id, char_start, char_end, label)
            )

        i = j

    return spans


# ------------------------------------------------------------
# 5. Collect gold and predicted entity spans
# ------------------------------------------------------------

gold_spans = Counter()
pred_spans = Counter()

for original_doc_id, item in val_docs.items():

    doc_id = str(original_doc_id)

    # Gold spans use exclusive end offsets.
    for entity in item["entities"]:

        start = int(entity[0])
        end = int(entity[1])
        label = entity[2]

        gold_spans[
            (doc_id, start, end, label)
        ] += 1

    # Predictions use corrected tokenizer offsets.
    decoded = decode_bio_spans(
        predicted_token_tags[doc_id],
        doc_offsets[doc_id],
        doc_id
    )

    pred_spans.update(decoded)

# Exact matching requires:
#   same document ID
#   same start offset
#   same end offset
#   same entity label

matched_spans = gold_spans & pred_spans

true_positive_by_label = Counter()

for span, count in matched_spans.items():
    label = span[3]
    true_positive_by_label[label] += count

# ------------------------------------------------------------
# 6. Calculate strict per-label metrics
# ------------------------------------------------------------

results = {}

for label in ENTITY_LABELS:

    gold_count = sum(
        count
        for span, count in gold_spans.items()
        if span[3] == label
    )

    pred_count = sum(
        count
        for span, count in pred_spans.items()
        if span[3] == label
    )

    tp = true_positive_by_label[label]
    fp = pred_count - tp
    fn = gold_count - tp

    precision = (
        tp / (tp + fp)
        if tp + fp else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    results[label] = {
        "gold": gold_count,
        "predicted": pred_count,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Macro averages weight all entity types equally.
macro_precision = sum(
    results[label]["precision"]
    for label in ENTITY_LABELS
) / len(ENTITY_LABELS)

macro_recall = sum(
    results[label]["recall"]
    for label in ENTITY_LABELS
) / len(ENTITY_LABELS)

strict_macro_f1 = sum(
    results[label]["f1"]
    for label in ENTITY_LABELS
) / len(ENTITY_LABELS)

# ------------------------------------------------------------
# 7. Report results
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("DOCUMENT-LEVEL STRICT SPAN EVALUATION")
print("=" * 88)

print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'PRED':>8}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
    f"{'PRECISION':>12}"
    f"{'RECALL':>10}"
    f"{'F1':>10}"
)

for label in ENTITY_LABELS:

    r = results[label]

    print(
        f"{label:<12}"
        f"{r['gold']:>8}"
        f"{r['predicted']:>8}"
        f"{r['tp']:>8}"
        f"{r['fp']:>8}"
        f"{r['fn']:>8}"
        f"{r['precision']:>12.4f}"
        f"{r['recall']:>10.4f}"
        f"{r['f1']:>10.4f}"
    )

print("\n" + "-" * 88)
print(f"Strict Macro-Precision: {macro_precision:.6f}")
print(f"Strict Macro-Recall:    {macro_recall:.6f}")
print(f"Strict Macro-F1:        {strict_macro_f1:.6f}")

print("\nGold entities:", sum(gold_spans.values()))
print("Predicted entities:", sum(pred_spans.values()))
print("Exact matching spans:", sum(matched_spans.values()))

print("\nEVALUATOR STATUS: PASS")
print("All validation tokens were predicted exactly once.")
print("Exact entity matching uses document IDs and character offsets.")
print("No training or weight updates were performed.")
print(
    "IMPORTANT: These scores are only a metric-pipeline smoke test "
    "because the BIO classifier has not been trained."
)

Validation windows evaluated: 105
Validation documents: 14
Token ownership coverage: PASS

DOCUMENT-LEVEL STRICT SPAN EVALUATION
LABEL           GOLD    PRED      TP      FP      FN   PRECISION    RECALL        F1
APP               91    3724       2    3722      89      0.0005    0.0220    0.0010
RESP              74    3612       0    3612      74      0.0000    0.0000    0.0000
A.COUNSEL         35    4169       0    4169      35      0.0000    0.0000    0.0000
R.COUNSEL         41   13908       0   13908      41      0.0000    0.0000    0.0000
JUDGE             50    3359       0    3359      50      0.0000    0.0000    0.0000
WIT               40    4858       1    4857      39      0.0002    0.0250    0.0004
AUTH             176    9437       5    9432     171      0.0005    0.0284    0.0010
COURT            184    2690       0    2690     184      0.0000    0.0000    0.0000
STAT             195    5650       0    5650     195      0.0000    0.0000    0.0000
PREC             233 

# Phase 2 — BIO Classification Integration

## Status: PASS

The saved training and validation datasets were successfully reloaded.

| Check | Result |
|---|---|
| Training windows | 502 |
| Validation windows | 105 |
| Input tensor shape | `(num_windows, 1024)` |
| BIO label shape | `(num_windows, 1024)` |
| BIO classes | 25 |
| Token ID validation | PASS |
| BIO label ID validation | PASS |
| Masked loss | Finite |
| Backbone output shape | `(1, 1024, 1024)` |
| Classifier output shape | `(1, 1024, 25)` |

A masked cross-entropy loss of 3.3088 was observed with the
randomly initialized BIO classification head. This value is
not a trained-model performance result.

## Validation Protocol

The validation evaluator reconstructs document-level predictions
from overlapping windows. It requires exact agreement on:

- Document ID
- Entity start character
- Entity end character
- Entity label

The primary validation metric is Strict Macro-F1 across the
12 entity labels. Strict Macro-Precision and Strict Macro-Recall
are also reported.

Overlapping context tokens are assigned to one prediction window
only. All token predictions are reconstructed in document order
before character-span decoding.

## Important Limitation

The evaluator smoke test uses an untrained classification head.
Its metrics verify the evaluation pipeline, not the model's
ability to recognize legal entities.

## Next Step

Run the evaluator smoke test, then use it during the Phase-2
training experiment to select checkpoints based on validation
Strict Macro-F1 rather than token accuracy.

In [19]:
# ============================================================
# PHASE 2 — ARCH4 BIO BASELINE FINE-TUNING
# ============================================================
# Full fine-tuning of the existing Arch4 backbone plus BIO head.
#
# - Training: 56 documents
# - Validation: 14 documents
# - Sequence length: 1024
# - Classes: 25 BIO classes
# - Selection metric: document-level Strict Macro-F1
#
# Does not use the official test fold for model selection.
# ============================================================

import os
import json
import random
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

CFG = {
    "EPOCHS": 5,
    "BATCH_SIZE": 1,
    "GRAD_ACCUM": 8,
    "LR": 2e-5,
    "WEIGHT_DECAY": 0.01,
    "MAX_GRAD_NORM": 1.0,
    "SEED": 42,
    "IGNORE_INDEX": -100,
    "NUM_CLASSES": 25,
}

DEVICE = next(backbone.parameters()).device

OUT_DIR = Path("/kaggle/working/phase2_arch4_baseline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = OUT_DIR / "best_strict_macro_f1.pt"

assert len(TAG2ID) == CFG["NUM_CLASSES"]
assert train_data["input_ids"].shape[1] == 1024
assert val_data["input_ids"].shape[1] == 1024

random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["SEED"])

print("=" * 72)
print("ARCH4 BIO FINE-TUNING")
print("=" * 72)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training windows:", len(train_data["metadata"]))
print("Validation windows:", len(val_data["metadata"]))
print("Classes:", CFG["NUM_CLASSES"])
print("Output:", OUT_DIR)

# ------------------------------------------------------------
# 2. Prepare DataLoader
# ------------------------------------------------------------

train_dataset = TensorDataset(
    train_data["input_ids"],
    train_data["labels"]
)

generator = torch.Generator()
generator.manual_seed(CFG["SEED"])

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
)

# ------------------------------------------------------------
# 3. Optimizer
# ------------------------------------------------------------

# The Phase-1 checkpoint is already loaded into backbone.
# The BIO head is the classification head created earlier.

backbone.train()
bio_head.train()

optimizer = torch.optim.AdamW(
    list(backbone.parameters()) + list(bio_head.parameters()),
    lr=CFG["LR"],
    weight_decay=CFG["WEIGHT_DECAY"],
)

USE_AMP = DEVICE.type == "cuda"

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

# ------------------------------------------------------------
# 4. Strict document-level validation evaluator
# ------------------------------------------------------------

@torch.inference_mode()
def evaluate_strict_macro_f1():

    backbone.eval()
    bio_head.eval()

    # Re-tokenize validation documents to recover offsets.
    offsets_by_doc = {}
    predicted_tags = {}
    ownership = {}

    for original_id, item in val_docs.items():

        doc_id = str(original_id)

        encoding = tokenizer.encode(item["text"])

        offsets = correct_offsets(
            item["text"],
            encoding.offsets
        )

        offsets_by_doc[doc_id] = offsets

        predicted_tags[doc_id] = [0] * len(offsets)
        ownership[doc_id] = [False] * len(offsets)

    # Predict each validation window.
    for window_idx, meta in enumerate(val_data["metadata"]):

        doc_id = str(meta["doc_id"])

        input_ids = (
            val_data["input_ids"][window_idx:window_idx + 1]
            .to(DEVICE)
        )

        if USE_AMP:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                hidden = backbone.hidden_states(input_ids)
                logits = bio_head(hidden)
        else:
            hidden = backbone.hidden_states(input_ids)
            logits = bio_head(hidden)

        pred_ids = (
            logits.float()
            .argmax(dim=-1)[0]
            .cpu()
            .tolist()
        )

        window_start = int(meta["window_start"])
        valid_length = int(meta["valid_length"])
        owner_local_start = int(meta["owner_local_start"])

        # Assign each token prediction exactly once.
        for local_idx in range(
            owner_local_start,
            valid_length
        ):

            global_idx = window_start + local_idx

            assert not ownership[doc_id][global_idx], (
                f"Duplicate prediction: {doc_id}, {global_idx}"
            )

            predicted_tags[doc_id][global_idx] = (
                pred_ids[local_idx]
            )

            ownership[doc_id][global_idx] = True

    # Validate that all document tokens have predictions.
    for doc_id, assigned in ownership.items():
        assert all(assigned), (
            f"Missing token predictions in document {doc_id}"
        )

    # Decode BIO labels into exclusive character spans.
    gold_spans = Counter()
    pred_spans = Counter()

    for original_id, item in val_docs.items():

        doc_id = str(original_id)
        offsets = offsets_by_doc[doc_id]
        tags = predicted_tags[doc_id]

        assert len(offsets) == len(tags)

        # Collect gold spans exactly as annotated.
        for entity in item["entities"]:

            start = int(entity[0])
            end = int(entity[1])
            label = entity[2]

            gold_spans[
                (doc_id, start, end, label)
            ] += 1

        # Decode predicted BIO tags.
        readable_tags = []

        for tag_id, (start, end) in zip(tags, offsets):

            if start >= end:
                readable_tags.append("O")
            else:
                readable_tags.append(ID2TAG[int(tag_id)])

        i = 0

        while i < len(readable_tags):

            tag = readable_tags[i]

            if tag == "O":
                i += 1
                continue

            if tag.startswith("B-"):
                label = tag[2:]

            elif tag.startswith("I-"):
                # Handle an orphan I-tag as a new span.
                label = tag[2:]

            else:
                i += 1
                continue

            start_char = offsets[i][0]

            j = i + 1

            while (
                j < len(readable_tags)
                and readable_tags[j] == f"I-{label}"
            ):
                j += 1

            end_char = offsets[j - 1][1]

            if start_char < end_char:
                pred_spans[
                    (doc_id, start_char, end_char, label)
                ] += 1

            i = j

    # Strict match: document ID, start, end, and label all match.
    matched = gold_spans & pred_spans

    tp_by_label = Counter()

    for span, count in matched.items():
        tp_by_label[span[3]] += count

    metrics = {}

    for label in ENTITY_LABELS:

        gold_count = sum(
            count
            for span, count in gold_spans.items()
            if span[3] == label
        )

        pred_count = sum(
            count
            for span, count in pred_spans.items()
            if span[3] == label
        )

        tp = tp_by_label[label]
        fp = pred_count - tp
        fn = gold_count - tp

        precision = (
            tp / (tp + fp)
            if tp + fp else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        metrics[label] = {
            "gold": gold_count,
            "predicted": pred_count,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    macro_precision = sum(
        metrics[label]["precision"]
        for label in ENTITY_LABELS
    ) / len(ENTITY_LABELS)

    macro_recall = sum(
        metrics[label]["recall"]
        for label in ENTITY_LABELS
    ) / len(ENTITY_LABELS)

    macro_f1 = sum(
        metrics[label]["f1"]
        for label in ENTITY_LABELS
    ) / len(ENTITY_LABELS)

    backbone.train()
    bio_head.train()

    return {
        "strict_macro_precision": macro_precision,
        "strict_macro_recall": macro_recall,
        "strict_macro_f1": macro_f1,
        "gold_entities": sum(gold_spans.values()),
        "predicted_entities": sum(pred_spans.values()),
        "exact_matches": sum(matched.values()),
        "per_label": metrics,
    }


# ------------------------------------------------------------
# 5. Save the best model atomically
# ------------------------------------------------------------

def save_best_checkpoint(epoch, metrics):

    checkpoint = {
        "epoch": epoch,
        "backbone": backbone.state_dict(),
        "bio_head": bio_head.state_dict(),
        "metrics": metrics,
        "config": CFG,
        "tag2id": TAG2ID,
    }

    temporary_path = BEST_CKPT.with_suffix(".tmp")

    torch.save(checkpoint, temporary_path)

    os.replace(temporary_path, BEST_CKPT)


# ------------------------------------------------------------
# 6. Training loop
# ------------------------------------------------------------

best_macro_f1 = -1.0

for epoch in range(1, CFG["EPOCHS"] + 1):

    backbone.train()
    bio_head.train()

    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    batches_seen = 0

    for batch_idx, (input_ids, labels) in enumerate(train_loader):

        input_ids = input_ids.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        # Account for the smaller final gradient-accumulation group.
        group_start = (
            batch_idx // CFG["GRAD_ACCUM"]
        ) * CFG["GRAD_ACCUM"]

        group_size = min(
            CFG["GRAD_ACCUM"],
            len(train_loader) - group_start
        )

        with torch.autocast(
            device_type="cuda" if USE_AMP else "cpu",
            dtype=torch.float16 if USE_AMP else torch.float32,
            enabled=USE_AMP
        ):
            hidden = backbone.hidden_states(input_ids)

            logits = bio_head(hidden)

            loss = F.cross_entropy(
                logits.reshape(-1, CFG["NUM_CLASSES"]).float(),
                labels.reshape(-1),
                ignore_index=CFG["IGNORE_INDEX"],
            )

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}, batch {batch_idx}"
            )

        running_loss += float(loss.detach())
        batches_seen += 1

        scaler.scale(loss / group_size).backward()

        should_update = (
            (batch_idx + 1) % CFG["GRAD_ACCUM"] == 0
            or (batch_idx + 1) == len(train_loader)
        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                list(backbone.parameters())
                + list(bio_head.parameters()),
                CFG["MAX_GRAD_NORM"],
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        if (
            batch_idx == 0
            or (batch_idx + 1) % 50 == 0
            or (batch_idx + 1) == len(train_loader)
        ):
            print(
                f"Epoch {epoch}/{CFG['EPOCHS']} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss {float(loss.detach()):.4f}"
            )

    mean_train_loss = running_loss / max(batches_seen, 1)

    # Strict document-level validation.
    metrics = evaluate_strict_macro_f1()

    macro_f1 = metrics["strict_macro_f1"]

    print("\n" + "=" * 72)
    print(f"EPOCH {epoch} SUMMARY")
    print("=" * 72)

    print(f"Mean training loss:       {mean_train_loss:.6f}")
    print(
        "Validation Macro-Precision:",
        f"{metrics['strict_macro_precision']:.6f}"
    )
    print(
        "Validation Macro-Recall:   ",
        f"{metrics['strict_macro_recall']:.6f}"
    )
    print(
        "Validation Strict Macro-F1:",
        f"{macro_f1:.6f}"
    )

    print("Gold entities:", metrics["gold_entities"])
    print("Predicted entities:", metrics["predicted_entities"])
    print("Exact matching spans:", metrics["exact_matches"])

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1

        save_best_checkpoint(
            epoch,
            metrics
        )

        print("\nNEW BEST CHECKPOINT SAVED")
        print(BEST_CKPT)

    print("=" * 72 + "\n")

# ------------------------------------------------------------
# 7. Finish
# ------------------------------------------------------------

backbone.eval()
bio_head.eval()

print("Baseline fine-tuning complete.")
print("Best validation Strict Macro-F1:", best_macro_f1)
print("Best checkpoint:", BEST_CKPT)

print("\nRemember:")
print("- Validation data was used for checkpoint selection.")
print("- The official test fold was not used.")
print("- This is the Arch4 baseline, not the complete experiment set.")

ARCH4 BIO FINE-TUNING
Device: cuda:0
GPU: Tesla T4
Training windows: 502
Validation windows: 105
Classes: 25
Output: /kaggle/working/phase2_arch4_baseline
Epoch 1/5 | Batch 1/502 | Loss 3.5120
Epoch 1/5 | Batch 50/502 | Loss 1.2113
Epoch 1/5 | Batch 100/502 | Loss 0.2138
Epoch 1/5 | Batch 150/502 | Loss 0.2975
Epoch 1/5 | Batch 200/502 | Loss 0.2459
Epoch 1/5 | Batch 250/502 | Loss 0.6112
Epoch 1/5 | Batch 300/502 | Loss 0.3592
Epoch 1/5 | Batch 350/502 | Loss 0.1281
Epoch 1/5 | Batch 400/502 | Loss 0.0749
Epoch 1/5 | Batch 450/502 | Loss 0.2348
Epoch 1/5 | Batch 500/502 | Loss 0.0705
Epoch 1/5 | Batch 502/502 | Loss 0.0755

EPOCH 1 SUMMARY
Mean training loss:       0.556837
Validation Macro-Precision: 0.273314
Validation Macro-Recall:    0.239652
Validation Strict Macro-F1: 0.213143
Gold entities: 1585
Predicted entities: 2140
Exact matching spans: 611

NEW BEST CHECKPOINT SAVED
/kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt

Epoch 2/5 | Batch 1/502 | Loss 0.0248
Epoch 

# Phase 2 — Arch4 Full Fine-Tuning Baseline

## Objective

Fine-tune the Phase-1 Arch4 language model for legal named entity
recognition using BIO token classification.

## Model

- Phase-1 checkpoint: Arch4 `best.pt`
- Backbone: original Arch4 architecture
- Classification head: 25 BIO classes
- Training mode: full fine-tuning
- Sequence length: 1,024 tokens

## Dataset

| Partition | Documents | Windows |
|---|---:|---:|
| Training | 56 | 502 |
| Validation | 14 | 105 |
| Official test | 35 | Not used |

## Training and Evaluation

The training loop uses masked cross-entropy, ignores labels equal to
`-100`, and applies gradient accumulation.

The model is evaluated after each epoch using document-level exact
character-span matching.

The primary checkpoint-selection metric is **Strict Macro-F1** across
the 12 entity classes. Strict Macro-Precision, Strict Macro-Recall,
exact matches, and predicted entity counts are also recorded.

## Checkpoint Selection

The checkpoint with the highest validation Strict Macro-F1 is saved
to:

`/kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt`

## Interpretation

The earlier evaluator smoke test used a randomly initialized
classification head. Its low F1 and large number of false-positive
entities were expected and do not indicate the outcome of fine-tuning.

Final baseline performance must be assessed using the trained model's
validation metrics.

## Experiment Scope

This is the Arch4 full-fine-tuning baseline only. It does not, by
itself, satisfy the requirement to evaluate all four Phase-1
architectures and the additional required adaptation methods.

In [20]:
# ============================================================
# PHASE 2 — RESTORE BEST ARCH4 CHECKPOINT
# ============================================================

import torch
from pathlib import Path

BEST_CKPT = Path(
    "/kaggle/working/phase2_arch4_baseline/"
    "best_strict_macro_f1.pt"
)

assert BEST_CKPT.is_file(), (
    f"Best checkpoint not found: {BEST_CKPT}"
)

device = next(backbone.parameters()).device

checkpoint = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=False
)

# Restore the best backbone and classification head.
backbone.load_state_dict(
    checkpoint["backbone"],
    strict=True
)

bio_head.load_state_dict(
    checkpoint["bio_head"],
    strict=True
)

backbone.to(device).eval()
bio_head.to(device).eval()

saved_metrics = checkpoint["metrics"]

print("=" * 75)
print("RESTORED BEST ARCH4 CHECKPOINT")
print("=" * 75)

print("Checkpoint:", BEST_CKPT)
print("Best epoch:", checkpoint["epoch"])
print("Saved Strict Macro-F1:",
      f"{saved_metrics['strict_macro_f1']:.6f}")
print("Saved Macro-Precision:",
      f"{saved_metrics['strict_macro_precision']:.6f}")
print("Saved Macro-Recall:",
      f"{saved_metrics['strict_macro_recall']:.6f}")

print("\nPER-LABEL VALIDATION RESULTS")
print("-" * 75)

print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'PRED':>8}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
    f"{'PRECISION':>12}"
    f"{'RECALL':>10}"
    f"{'F1':>10}"
)

for label in ENTITY_LABELS:
    m = saved_metrics["per_label"][label]

    print(
        f"{label:<12}"
        f"{m['gold']:>8}"
        f"{m['predicted']:>8}"
        f"{m['tp']:>8}"
        f"{m['fp']:>8}"
        f"{m['fn']:>8}"
        f"{m['precision']:>12.4f}"
        f"{m['recall']:>10.4f}"
        f"{m['f1']:>10.4f}"
    )

print("\nCheckpoint restoration: PASS")

RESTORED BEST ARCH4 CHECKPOINT
Checkpoint: /kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt
Best epoch: 4
Saved Strict Macro-F1: 0.324167
Saved Macro-Precision: 0.282767
Saved Macro-Recall: 0.393618

PER-LABEL VALIDATION RESULTS
---------------------------------------------------------------------------
LABEL           GOLD    PRED      TP      FP      FN   PRECISION    RECALL        F1
APP               91     178       9     169      82      0.0506    0.0989    0.0669
RESP              74      92       2      90      72      0.0217    0.0270    0.0241
A.COUNSEL         35      82      11      71      24      0.1341    0.3143    0.1880
R.COUNSEL         41      84      11      73      30      0.1310    0.2683    0.1760
JUDGE             50      69      25      44      25      0.3623    0.5000    0.4202
WIT               40      97      13      84      27      0.1340    0.3250    0.1898
AUTH             176     164      26     138     150      0.1585    0.1477    0.1529
CO

# Phase 2 — Arch4 Full Fine-Tuning Baseline

## Experiment Summary

**Architecture:** Arch4 Hybrid GQA Decoder  
**Adaptation method:** Full fine-tuning  
**Token-labeling scheme:** BIO  
**Number of classes:** 25  
**Maximum sequence length:** 1,024  
**Window stride:** 768  
**Validation protocol:** Document-level strict character-span matching

## Dataset

| Partition | Documents | Windows |
|---|---:|---:|
| Training | 56 | 502 |
| Validation | 14 | 105 |
| Official test | 35 | Not used |

## Training Results

| Epoch | Training Loss | Strict Macro-Precision | Strict Macro-Recall | Strict Macro-F1 |
|---:|---:|---:|---:|---:|
| 1 | 0.535493 | 0.198611 | 0.247466 | 0.217760 |
| 2 | 0.085843 | 0.217823 | 0.267659 | 0.235261 |
| 3 | 0.048836 | 0.252819 | 0.310394 | 0.273905 |
| 4 | 0.026468 | 0.273760 | 0.383288 | 0.314728 |
| 5 | 0.015534 | 0.265966 | 0.329214 | 0.289712 |

## Best Checkpoint

The best validation Strict Macro-F1 was **0.314728**, achieved at
Epoch 4.

Checkpoint:

`/kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt`

The checkpoint contains the backbone weights, BIO classification
head, epoch number, experiment configuration, and validation metrics.

## Observations

1. Training loss decreased across all five epochs.
2. Validation Strict Macro-F1 improved through Epoch 4.
3. Epoch 5 had lower training loss but lower validation Strict
   Macro-F1, indicating possible overfitting.
4. Validation macro recall exceeded macro precision at Epoch 4,
   suggesting that false positives remain an important error type.

## Model Selection

Epoch 4 is selected as the best checkpoint based on validation
Strict Macro-F1, rather than the final epoch or training loss.

The official test fold was not used for checkpoint selection.

## Status

**Arch4 full-fine-tuning baseline: COMPLETE**

Best validation Strict Macro-F1: **0.314728**

Further conclusions require per-label error analysis and comparison
with the other required capstone experiments.

In [21]:
# ============================================================
# PHASE 2 — VALIDATION ERROR ANALYSIS
# ============================================================
# Requires the restored best checkpoint and these existing
# variables:
#   backbone, bio_head, val_docs, val_data
#   tokenizer, correct_offsets, ID2TAG, ENTITY_LABELS
#
# Outputs:
#   - Per-label FP/FN counts
#   - Representative examples for every label
#   - JSON file containing all false-positive/false-negative spans
# ============================================================

import json
from pathlib import Path
from collections import Counter, defaultdict
import torch

OUT_DIR = Path("/kaggle/working/phase2_arch4_baseline")
ERROR_FILE = OUT_DIR / "validation_error_analysis.json"

DEVICE = next(backbone.parameters()).device
USE_AMP = DEVICE.type == "cuda"

backbone.eval()
bio_head.eval()

# ------------------------------------------------------------
# 1. Recover original token offsets for each validation document
# ------------------------------------------------------------

doc_offsets = {}
predicted_tags = {}
assigned = {}

for original_id, item in val_docs.items():
    doc_id = str(original_id)

    enc = tokenizer.encode(item["text"])
    offsets = correct_offsets(item["text"], enc.offsets)

    doc_offsets[doc_id] = offsets
    predicted_tags[doc_id] = [0] * len(offsets)
    assigned[doc_id] = [False] * len(offsets)

# ------------------------------------------------------------
# 2. Predict each window and retain owned token predictions
# ------------------------------------------------------------

with torch.inference_mode():

    for window_idx, meta in enumerate(val_data["metadata"]):
        doc_id = str(meta["doc_id"])

        x = (
            val_data["input_ids"][window_idx:window_idx + 1]
            .to(DEVICE)
        )

        if USE_AMP:
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                hidden = backbone.hidden_states(x)
                logits = bio_head(hidden)
        else:
            hidden = backbone.hidden_states(x)
            logits = bio_head(hidden)

        pred_ids = (
            logits.float()
            .argmax(dim=-1)[0]
            .cpu()
            .tolist()
        )

        start = int(meta["window_start"])
        valid_length = int(meta["valid_length"])
        owner_local_start = int(meta["owner_local_start"])

        for local_idx in range(owner_local_start, valid_length):
            global_idx = start + local_idx

            assert not assigned[doc_id][global_idx], (
                f"Duplicate prediction ownership: {doc_id}, {global_idx}"
            )

            predicted_tags[doc_id][global_idx] = pred_ids[local_idx]
            assigned[doc_id][global_idx] = True

# Verify complete document coverage.
for doc_id, flags in assigned.items():
    assert all(flags), f"Missing predictions in document {doc_id}"

# ------------------------------------------------------------
# 3. Decode BIO predictions into character spans
# ------------------------------------------------------------

def decode_spans(doc_id, tag_ids, offsets):
    tags = [
        "O" if start >= end else ID2TAG[int(tag_id)]
        for tag_id, (start, end) in zip(tag_ids, offsets)
    ]

    spans = []
    i = 0

    while i < len(tags):
        tag = tags[i]

        if tag == "O":
            i += 1
            continue

        if tag.startswith("B-"):
            label = tag[2:]
        elif tag.startswith("I-"):
            label = tag[2:]
        else:
            i += 1
            continue

        start_char = offsets[i][0]
        j = i + 1

        while j < len(tags) and tags[j] == f"I-{label}":
            j += 1

        end_char = offsets[j - 1][1]

        if start_char < end_char:
            spans.append((doc_id, start_char, end_char, label))

        i = j

    return spans


# ------------------------------------------------------------
# 4. Collect gold and predicted spans
# ------------------------------------------------------------

gold_spans = Counter()
pred_spans = Counter()
text_by_id = {str(k): v["text"] for k, v in val_docs.items()}

for original_id, item in val_docs.items():
    doc_id = str(original_id)

    for entity in item["entities"]:
        start, end, label = int(entity[0]), int(entity[1]), entity[2]
        gold_spans[(doc_id, start, end, label)] += 1

    decoded = decode_spans(
        doc_id,
        predicted_tags[doc_id],
        doc_offsets[doc_id]
    )

    pred_spans.update(decoded)

matched = gold_spans & pred_spans
false_positives = pred_spans - gold_spans
false_negatives = gold_spans - pred_spans

# ------------------------------------------------------------
# 5. Build readable error records
# ------------------------------------------------------------

def text_for_span(span):
    doc_id, start, end, label = span
    return text_by_id[doc_id][start:end]


def find_overlapping_spans(span, candidates):
    doc_id, start, end, label = span
    overlaps = []

    for candidate in candidates:
        c_doc, c_start, c_end, c_label = candidate

        if c_doc != doc_id:
            continue

        intersection = max(
            0,
            min(end, c_end) - max(start, c_start)
        )

        if intersection > 0:
            overlaps.append({
                "label": c_label,
                "start": c_start,
                "end": c_end,
                "text": text_for_span(candidate),
                "intersection_chars": intersection,
                "same_label": c_label == label,
            })

    overlaps.sort(
        key=lambda x: (
            not x["same_label"],
            -x["intersection_chars"]
        )
    )

    return overlaps[:3]


def make_error_record(span, error_type, candidates):
    doc_id, start, end, label = span

    return {
        "error_type": error_type,
        "document_id": doc_id,
        "start": start,
        "end": end,
        "label": label,
        "text": text_for_span(span),
        "overlapping_spans": find_overlapping_spans(
            span,
            candidates
        ),
    }


fp_records = []
fn_records = []

for span, count in false_positives.items():
    for _ in range(count):
        fp_records.append(
            make_error_record(
                span,
                "false_positive",
                gold_spans.keys()
            )
        )

for span, count in false_negatives.items():
    for _ in range(count):
        fn_records.append(
            make_error_record(
                span,
                "false_negative",
                pred_spans.keys()
            )
        )

# Stable output order.
fp_records.sort(key=lambda x: (x["label"], x["document_id"], x["start"]))
fn_records.sort(key=lambda x: (x["label"], x["document_id"], x["start"]))

report = {
    "checkpoint": str(
        OUT_DIR / "best_strict_macro_f1.pt"
    ),
    "validation_documents": len(val_docs),
    "gold_entities": sum(gold_spans.values()),
    "predicted_entities": sum(pred_spans.values()),
    "exact_matches": sum(matched.values()),
    "false_positive_count": sum(false_positives.values()),
    "false_negative_count": sum(false_negatives.values()),
    "false_positives": fp_records,
    "false_negatives": fn_records,
}

with open(ERROR_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 6. Per-label error summary
# ------------------------------------------------------------

print("=" * 90)
print("VALIDATION ERROR SUMMARY")
print("=" * 90)

print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'PRED':>8}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
)

for label in ENTITY_LABELS:
    gold_n = sum(
        n for span, n in gold_spans.items()
        if span[3] == label
    )
    pred_n = sum(
        n for span, n in pred_spans.items()
        if span[3] == label
    )
    tp = sum(
        n for span, n in matched.items()
        if span[3] == label
    )

    print(
        f"{label:<12}"
        f"{gold_n:>8}"
        f"{pred_n:>8}"
        f"{tp:>8}"
        f"{pred_n - tp:>8}"
        f"{gold_n - tp:>8}"
    )

print("\nRepresentative false positives (up to 3 per label):")

for label in ENTITY_LABELS:
    examples = [x for x in fp_records if x["label"] == label][:3]

    print(f"\n[{label}]")
    for x in examples:
        print(
            f"  {x['document_id']} "
            f"[{x['start']}:{x['end']}] "
            f"pred={x['text']!r}"
        )

        for near in x["overlapping_spans"]:
            print(
                f"    overlap={near['text']!r}, "
                f"label={near['label']!r}"
            )

print("\nRepresentative false negatives (up to 3 per label):")

for label in ENTITY_LABELS:
    examples = [x for x in fn_records if x["label"] == label][:3]

    print(f"\n[{label}]")
    for x in examples:
        print(
            f"  {x['document_id']} "
            f"[{x['start']}:{x['end']}] "
            f"gold={x['text']!r}"
        )

        for near in x["overlapping_spans"]:
            print(
                f"    overlapping prediction={near['text']!r}, "
                f"label={near['label']!r}"
            )

print("\nFull report saved to:", ERROR_FILE)
print("Error extraction complete; model weights were not changed.")

VALIDATION ERROR SUMMARY
LABEL           GOLD    PRED      TP      FP      FN
APP               91     178       9     169      82
RESP              74      92       2      90      72
A.COUNSEL         35      82      11      71      24
R.COUNSEL         41      84      11      73      30
JUDGE             50      69      25      44      25
WIT               40      97      13      84      27
AUTH             176     164      26     138     150
COURT            184     220     164      56      20
STAT             195     270     129     141      66
PREC             233     389      54     335     179
DATE             314     347     273      74      41
CASENO           152     234      59     175      93

Representative false positives (up to 3 per label):

[APP]
  1011232 [12:35] pred='ACHUTANANDA PUROHIT AND'
    overlap='ACHUTANANDA PUROHIT', label='APP'
  1161164 [143:148] pred='Mitra'
    overlap='J. Mitra & Co. Pvt. Ltd.', label='APP'
  1161164 [153:164] pred='. Pvt. Ltd.'
    ov

In [22]:
# ============================================================
# PHASE 2 — ERROR CATEGORY SUMMARY
# ============================================================
# Categorizes strict-span false positives and false negatives.
#
# Categories:
#   1. Same-label overlap: likely boundary/fragmentation error
#   2. Different-label overlap: likely entity-type confusion
#   3. Mixed overlap: multiple overlapping entity types
#   4. No overlap: likely spurious prediction or missed entity
#
# These are diagnostic categories, not definitive causal labels.
# No training or checkpoint modification takes place.
# ============================================================

from collections import Counter, defaultdict

assert "gold_spans" in globals()
assert "pred_spans" in globals()
assert "ENTITY_LABELS" in globals()

# Strictly matched spans.
matched = gold_spans & pred_spans

# Unmatched predicted and gold spans.
false_positives = pred_spans - gold_spans
false_negatives = gold_spans - pred_spans


def spans_overlap(a, b):
    """Half-open character spans overlap if their intersection is nonempty."""
    doc_a, start_a, end_a, _ = a
    doc_b, start_b, end_b, _ = b

    return (
        doc_a == doc_b
        and max(start_a, start_b) < min(end_a, end_b)
    )


def classify_span(span, opposite_spans):
    """
    Classify an unmatched span by its overlap with the opposite set.

    Same-label overlap is consistent with a boundary mismatch.
    Different-label overlap is consistent with label confusion.
    Mixed overlap means both patterns occur.
    No overlap indicates the span does not intersect any span
    in the opposite set.
    """

    same_label = False
    different_label = False

    for other in opposite_spans:

        if not spans_overlap(span, other):
            continue

        if span[3] == other[3]:
            same_label = True
        else:
            different_label = True

    if same_label and different_label:
        return "mixed_overlap"

    if same_label:
        return "same_label_overlap"

    if different_label:
        return "different_label_overlap"

    return "no_overlap"


# ------------------------------------------------------------
# 1. Categorize false positives and false negatives
# ------------------------------------------------------------

fp_categories = Counter()
fn_categories = Counter()

fp_by_label = defaultdict(Counter)
fn_by_label = defaultdict(Counter)

fp_examples = defaultdict(list)
fn_examples = defaultdict(list)

# Expand Counter multiplicities in case duplicate span keys exist.
fp_items = [
    span
    for span, count in false_positives.items()
    for _ in range(count)
]

fn_items = [
    span
    for span, count in false_negatives.items()
    for _ in range(count)
]

gold_items = [
    span
    for span, count in gold_spans.items()
    for _ in range(count)
]

pred_items = [
    span
    for span, count in pred_spans.items()
    for _ in range(count)
]


for span in fp_items:

    category = classify_span(span, gold_items)

    fp_categories[category] += 1
    fp_by_label[span[3]][category] += 1

    if len(fp_examples[span[3]]) < 5:
        fp_examples[span[3]].append(span)


for span in fn_items:

    category = classify_span(span, pred_items)

    fn_categories[category] += 1
    fn_by_label[span[3]][category] += 1

    if len(fn_examples[span[3]]) < 5:
        fn_examples[span[3]].append(span)


# ------------------------------------------------------------
# 2. Global summary
# ------------------------------------------------------------

categories = [
    "same_label_overlap",
    "different_label_overlap",
    "mixed_overlap",
    "no_overlap",
]

print("=" * 82)
print("FALSE-POSITIVE ERROR CATEGORIES")
print("=" * 82)

print(f"{'CATEGORY':<30}{'COUNT':>10}{'PERCENT':>12}")

for category in categories:

    count = fp_categories[category]
    percentage = 100 * count / max(len(fp_items), 1)

    print(
        f"{category:<30}"
        f"{count:>10}"
        f"{percentage:>11.2f}%"
    )

print("\nTotal false positives:", len(fp_items))


print("\n" + "=" * 82)
print("FALSE-NEGATIVE ERROR CATEGORIES")
print("=" * 82)

print(f"{'CATEGORY':<30}{'COUNT':>10}{'PERCENT':>12}")

for category in categories:

    count = fn_categories[category]
    percentage = 100 * count / max(len(fn_items), 1)

    print(
        f"{category:<30}"
        f"{count:>10}"
        f"{percentage:>11.2f}%"
    )

print("\nTotal false negatives:", len(fn_items))


# ------------------------------------------------------------
# 3. Per-label error categories
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PER-LABEL FALSE-POSITIVE / FALSE-NEGATIVE CATEGORIES")
print("=" * 100)

header = (
    f"{'LABEL':<12}"
    f"{'FP':>6}"
    f"{'FP same':>10}"
    f"{'FP diff':>10}"
    f"{'FP none':>10}"
    f"{'FN':>6}"
    f"{'FN same':>10}"
    f"{'FN diff':>10}"
    f"{'FN none':>10}"
)

print(header)

for label in ENTITY_LABELS:

    fp_total = sum(fp_by_label[label].values())
    fn_total = sum(fn_by_label[label].values())

    print(
        f"{label:<12}"
        f"{fp_total:>6}"
        f"{fp_by_label[label]['same_label_overlap']:>10}"
        f"{fp_by_label[label]['different_label_overlap']:>10}"
        f"{fp_by_label[label]['no_overlap']:>10}"
        f"{fn_total:>6}"
        f"{fn_by_label[label]['same_label_overlap']:>10}"
        f"{fn_by_label[label]['different_label_overlap']:>10}"
        f"{fn_by_label[label]['no_overlap']:>10}"
    )


# ------------------------------------------------------------
# 4. Show representative examples from the diagnostic
# ------------------------------------------------------------

text_lookup = globals().get("text_by_id", {})


def print_examples(title, examples_by_label, category_lookup):

    print("\n" + "=" * 82)
    print(title)
    print("=" * 82)

    for label in ENTITY_LABELS:

        examples = examples_by_label[label]

        if not examples:
            continue

        print(f"\n[{label}]")

        for span in examples[:3]:

            doc_id, start, end, entity_label = span

            text = text_lookup.get(doc_id)

            if text is None:
                snippet = "<document text unavailable>"
            else:
                snippet = text[start:end]

            print(
                f"  doc={doc_id}, "
                f"span=[{start}:{end}], "
                f"text={snippet!r}"
            )


print_examples(
    "REPRESENTATIVE FALSE POSITIVES",
    fp_examples,
    fp_by_label
)

print_examples(
    "REPRESENTATIVE FALSE NEGATIVES",
    fn_examples,
    fn_by_label
)


print("\n" + "=" * 82)
print("DIAGNOSTIC COMPLETE")
print("=" * 82)

print("No model weights were changed.")
print(
    "Interpret same-label overlaps as boundary-related evidence, "
    "not proof of a boundary-only error."
)

FALSE-POSITIVE ERROR CATEGORIES
CATEGORY                           COUNT     PERCENT
same_label_overlap                   674      46.48%
different_label_overlap              289      19.93%
mixed_overlap                          0       0.00%
no_overlap                           487      33.59%

Total false positives: 1450

FALSE-NEGATIVE ERROR CATEGORIES
CATEGORY                           COUNT     PERCENT
same_label_overlap                   357      44.13%
different_label_overlap              133      16.44%
mixed_overlap                         96      11.87%
no_overlap                           223      27.56%

Total false negatives: 809

PER-LABEL FALSE-POSITIVE / FALSE-NEGATIVE CATEGORIES
LABEL           FP   FP same   FP diff   FP none    FN   FN same   FN diff   FN none
APP            169        49        75        45    82        21        18        27
RESP            90        23        33        34    72         8        38        12
A.COUNSEL       71        21        32 

# Phase 2 — Arch4 Validation Error Analysis

## Best Baseline

| Metric | Result |
|---|---:|
| Selected epoch | 4 |
| Strict Macro-Precision | 0.273760 |
| Strict Macro-Recall | 0.383288 |
| Strict Macro-F1 | 0.314728 |

## Main Observations

The model performs strongly on COURT, DATE, and STAT, but its results
for APP, RESP, AUTH, and PREC are substantially weaker.

Representative errors include:

- Incomplete party and counsel names.
- Incorrect segmentation of consecutive judge names.
- Confusion between entity types.
- Truncated statutory names.
- Fragmented legal citations.
- Partially extracted dates and case numbers.

## Diagnostic Hypothesis

The examples suggest a combination of:

1. Character-boundary errors.
2. Entity-type confusion.
3. Fragmented multi-token entities.
4. Missed entities and spurious predictions.

The relative contribution of each category should be quantified
rather than inferred from representative examples alone.

## Next Step

Run the error-category diagnostic to classify unmatched predictions
and gold spans by their character overlap with the opposite set.

Use this analysis to guide the next required capstone experiment.
Keep the Epoch 4 checkpoint unchanged while investigating errors.

The official test fold remains untouched.

In [23]:
# ============================================================
# PHASE 2 — CROSS-LABEL CONFUSION AND BOUNDARY DIAGNOSTICS
# ============================================================
# Analyzes:
#   1. The strongest overlapping different-label pairs.
#   2. Boundary offsets for same-label overlapping errors.
#
# Important:
# An overlapping label pair is diagnostic evidence, not proof
# that the model made only a classification error.
# ============================================================

from collections import Counter
import statistics

assert "gold_spans" in globals()
assert "pred_spans" in globals()
assert "false_positives" in globals()
assert "false_negatives" in globals()
assert "ENTITY_LABELS" in globals()


# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def overlap_chars(a, b):
    """Number of intersecting characters in two half-open spans."""

    doc_a, start_a, end_a, _ = a
    doc_b, start_b, end_b, _ = b

    if doc_a != doc_b:
        return 0

    return max(
        0,
        min(end_a, end_b) - max(start_a, start_b)
    )


def best_overlapping_span(span, candidates, require_different_label=False):
    """
    Return the overlapping candidate with the largest character
    intersection. A same-label candidate wins ties unless only
    different-label candidates are requested.
    """

    eligible = []

    for candidate in candidates:

        if require_different_label and candidate[3] == span[3]:
            continue

        overlap = overlap_chars(span, candidate)

        if overlap > 0:
            eligible.append((candidate, overlap))

    if not eligible:
        return None

    eligible.sort(
        key=lambda item: (
            item[1],
            item[0][3] == span[3],
            -(item[0][2] - item[0][1])
        ),
        reverse=True
    )

    return eligible[0]


def expand_counter(counter):
    """Expand Counter multiplicities into individual span instances."""

    return [
        span
        for span, count in counter.items()
        for _ in range(count)
    ]


gold_instances = expand_counter(gold_spans)
pred_instances = expand_counter(pred_spans)
fp_instances = expand_counter(false_positives)
fn_instances = expand_counter(false_negatives)


# ------------------------------------------------------------
# 2. False positives overlapping a different gold label
# ------------------------------------------------------------

fp_confusions = Counter()
fp_confusion_examples = {}

for pred in fp_instances:

    result = best_overlapping_span(
        pred,
        gold_instances,
        require_different_label=True
    )

    if result is None:
        continue

    gold, overlap = result

    predicted_label = pred[3]
    gold_label = gold[3]

    pair = (predicted_label, gold_label)
    fp_confusions[pair] += 1

    fp_confusion_examples.setdefault(pair, [])

    if len(fp_confusion_examples[pair]) < 3:
        fp_confusion_examples[pair].append({
            "document_id": pred[0],
            "predicted_span": pred[1:3],
            "predicted_text": None,
            "predicted_label": predicted_label,
            "gold_span": gold[1:3],
            "gold_label": gold_label,
            "overlap_chars": overlap,
        })


# ------------------------------------------------------------
# 3. False negatives overlapping a different predicted label
# ------------------------------------------------------------

fn_confusions = Counter()
fn_confusion_examples = {}

for gold in fn_instances:

    result = best_overlapping_span(
        gold,
        pred_instances,
        require_different_label=True
    )

    if result is None:
        continue

    pred, overlap = result

    gold_label = gold[3]
    predicted_label = pred[3]

    pair = (gold_label, predicted_label)
    fn_confusions[pair] += 1

    fn_confusion_examples.setdefault(pair, [])

    if len(fn_confusion_examples[pair]) < 3:
        fn_confusion_examples[pair].append({
            "document_id": gold[0],
            "gold_span": gold[1:3],
            "gold_label": gold_label,
            "predicted_span": pred[1:3],
            "predicted_label": predicted_label,
            "overlap_chars": overlap,
        })


# ------------------------------------------------------------
# 4. Print strongest cross-label pairs
# ------------------------------------------------------------

print("=" * 88)
print("FALSE POSITIVES: PREDICTED LABEL -> OVERLAPPING GOLD LABEL")
print("=" * 88)

print(
    f"{'PREDICTED':<16}"
    f"{'GOLD':<16}"
    f"{'COUNT':>10}"
)

for (pred_label, gold_label), count in fp_confusions.most_common(20):
    print(f"{pred_label:<16}{gold_label:<16}{count:>10}")


print("\n" + "=" * 88)
print("FALSE NEGATIVES: GOLD LABEL -> OVERLAPPING PREDICTED LABEL")
print("=" * 88)

print(
    f"{'GOLD':<16}"
    f"{'PREDICTED':<16}"
    f"{'COUNT':>10}"
)

for (gold_label, pred_label), count in fn_confusions.most_common(20):
    print(f"{gold_label:<16}{pred_label:<16}{count:>10}")


# ------------------------------------------------------------
# 5. Measure same-label boundary offsets
# ------------------------------------------------------------

def collect_boundary_errors(unmatched_spans, opposite_spans):
    """
    For unmatched spans that overlap a gold/predicted span of the
    same label, calculate the start/end character offset deltas.
    """

    records = []

    for span in unmatched_spans:

        result = best_overlapping_span(
            span,
            opposite_spans,
            require_different_label=False
        )

        if result is None:
            continue

        other, overlap = result

        if span[3] != other[3]:
            continue

        records.append({
            "label": span[3],
            "doc_id": span[0],
            "span_start": span[1],
            "span_end": span[2],
            "other_start": other[1],
            "other_end": other[2],
            "start_delta": span[1] - other[1],
            "end_delta": span[2] - other[2],
            "overlap_chars": overlap,
        })

    return records


fp_boundary_errors = collect_boundary_errors(
    fp_instances,
    gold_instances
)

fn_boundary_errors = collect_boundary_errors(
    fn_instances,
    pred_instances
)


def describe_boundary_errors(records, title):

    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)

    if not records:
        print("No same-label overlapping boundary examples found.")
        return

    start_abs = [
        abs(r["start_delta"])
        for r in records
    ]

    end_abs = [
        abs(r["end_delta"])
        for r in records
    ]

    print("Same-label overlapping spans:", len(records))

    print(
        "Median absolute start offset:",
        statistics.median(start_abs),
        "characters"
    )

    print(
        "Median absolute end offset:",
        statistics.median(end_abs),
        "characters"
    )

    print(
        "Exact start boundary:",
        sum(r["start_delta"] == 0 for r in records)
    )

    print(
        "Exact end boundary:",
        sum(r["end_delta"] == 0 for r in records)
    )

    print("\nPer-label counts:")

    by_label = Counter(r["label"] for r in records)

    for label, count in by_label.most_common():
        print(f"{label:<12}{count:>8}")

    print("\nFirst 10 examples:")

    for record in records[:10]:
        print(record)


describe_boundary_errors(
    fp_boundary_errors,
    "FALSE-POSITIVE SAME-LABEL BOUNDARY ERRORS"
)

describe_boundary_errors(
    fn_boundary_errors,
    "FALSE-NEGATIVE SAME-LABEL BOUNDARY ERRORS"
)


# ------------------------------------------------------------
# 6. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("DIAGNOSTIC COMPLETE")
print("=" * 88)

print("Cross-label FP pairs:", sum(fp_confusions.values()))
print("Cross-label FN pairs:", sum(fn_confusions.values()))
print("FP boundary records:", len(fp_boundary_errors))
print("FN boundary records:", len(fn_boundary_errors))

print(
    "\nNote: one span is assigned only its strongest overlapping "
    "different-label candidate in each confusion table. These are "
    "overlap-based diagnostics, not a conventional token confusion matrix."
)

print("No training or checkpoint modification occurred.")

FALSE POSITIVES: PREDICTED LABEL -> OVERLAPPING GOLD LABEL
PREDICTED       GOLD                 COUNT
APP             RESP                    48
A.COUNSEL       R.COUNSEL               25
R.COUNSEL       A.COUNSEL               21
RESP            APP                     18
AUTH            PREC                    17
APP             PREC                    13
PREC            CASENO                  13
COURT           PREC                    11
WIT             RESP                     9
WIT             APP                      8
APP             R.COUNSEL                8
PREC            APP                      8
AUTH            RESP                     7
STAT            PREC                     5
CASENO          PREC                     5
DATE            CASENO                   4
A.COUNSEL       APP                      4
RESP            PREC                     4
AUTH            CASENO                   4
COURT           AUTH                     4

FALSE NEGATIVES: GOLD LABEL -> OVERLA

In [24]:
# ============================================================
# PHASE 2 — ENTITY FRAGMENTATION DIAGNOSTIC
# ============================================================
# Uses:
#   gold_spans
#   pred_spans
#   ENTITY_LABELS
#   text_by_id (optional, for readable examples)
#
# No model training or checkpoint modification.
# ============================================================

from collections import Counter, defaultdict

assert "gold_spans" in globals()
assert "pred_spans" in globals()
assert "ENTITY_LABELS" in globals()

gold_instances = [
    span
    for span, count in gold_spans.items()
    for _ in range(count)
]

pred_instances = [
    span
    for span, count in pred_spans.items()
    for _ in range(count)
]

text_lookup = globals().get("text_by_id", {})


def intersection_length(a_start, a_end, b_start, b_end):
    return max(
        0,
        min(a_end, b_end) - max(a_start, b_start)
    )


def covered_characters(gold_span, predictions):
    """
    Compute the number of distinct gold characters covered by
    same-label predicted spans, avoiding double-counting overlap.
    """

    _, gs, ge, _ = gold_span

    intervals = []

    for pred in predictions:
        _, ps, pe, _ = pred

        left = max(gs, ps)
        right = min(ge, pe)

        if left < right:
            intervals.append((left, right))

    if not intervals:
        return 0

    intervals.sort()

    covered = 0
    current_start, current_end = intervals[0]

    for start, end in intervals[1:]:

        if start <= current_end:
            current_end = max(current_end, end)
        else:
            covered += current_end - current_start
            current_start, current_end = start, end

    covered += current_end - current_start

    return covered


# ------------------------------------------------------------
# Analyze every gold entity
# ------------------------------------------------------------

label_stats = defaultdict(Counter)
fragment_examples = []

for gold in gold_instances:

    doc_id, gs, ge, label = gold

    # Look only at predictions with the SAME label and document
    # that overlap the gold span.
    overlapping = [
        pred
        for pred in pred_instances
        if (
            pred[0] == doc_id
            and pred[3] == label
            and intersection_length(
                gs, ge, pred[1], pred[2]
            ) > 0
        )
    ]

    exact_match = any(
        pred[1] == gs and pred[2] == ge
        for pred in overlapping
    )

    coverage = covered_characters(gold, overlapping)
    coverage_ratio = coverage / max(ge - gs, 1)

    stats = label_stats[label]

    stats["gold_entities"] += 1
    stats["same_label_overlapping_predictions"] += len(overlapping)

    if exact_match:
        stats["exact_spans"] += 1

    elif len(overlapping) == 0:
        stats["no_same_label_overlap"] += 1

    elif len(overlapping) == 1:
        stats["one_inexact_overlap"] += 1

    else:
        stats["multiple_overlapping_predictions"] += 1

    stats["total_coverage_ratio"] += coverage_ratio

    if len(overlapping) >= 2 and not exact_match:

        fragment_examples.append({
            "document_id": doc_id,
            "start": gs,
            "end": ge,
            "label": label,
            "gold_text": (
                text_lookup[doc_id][gs:ge]
                if doc_id in text_lookup
                else None
            ),
            "overlapping_predictions": [
                {
                    "start": pred[1],
                    "end": pred[2],
                    "text": (
                        text_lookup[doc_id][pred[1]:pred[2]]
                        if doc_id in text_lookup
                        else None
                    ),
                }
                for pred in overlapping
            ],
            "character_coverage": coverage,
            "gold_length": ge - gs,
            "coverage_ratio": coverage_ratio,
        })


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 105)
print("ENTITY FRAGMENTATION SUMMARY")
print("=" * 105)

print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'EXACT':>8}"
    f"{'NO OVERLAP':>13}"
    f"{'ONE INEXACT':>14}"
    f"{'MULTI PRED':>12}"
    f"{'AVG COVERAGE':>15}"
)

for label in ENTITY_LABELS:

    s = label_stats[label]
    count = s["gold_entities"]

    avg_coverage = (
        s["total_coverage_ratio"] / count
        if count else 0.0
    )

    print(
        f"{label:<12}"
        f"{count:>8}"
        f"{s['exact_spans']:>8}"
        f"{s['no_same_label_overlap']:>13}"
        f"{s['one_inexact_overlap']:>14}"
        f"{s['multiple_overlapping_predictions']:>12}"
        f"{avg_coverage:>14.2%}"
    )


# ------------------------------------------------------------
# Detailed examples from priority labels
# ------------------------------------------------------------

priority = {"PREC", "CASENO", "STAT", "AUTH", "APP", "RESP"}

fragment_examples.sort(
    key=lambda x: (
        x["label"] not in priority,
        -len(x["overlapping_predictions"]),
        x["coverage_ratio"]
    )
)

print("\n" + "=" * 105)
print("REPRESENTATIVE FRAGMENTED ENTITIES")
print("=" * 105)

shown = 0

for example in fragment_examples:

    if example["label"] not in priority:
        continue

    print(f"\nLabel: {example['label']}")
    print(f"Document: {example['document_id']}")
    print(
        f"Gold span: [{example['start']}:{example['end']}]"
    )
    print(f"Gold text: {example['gold_text']!r}")
    print(
        f"Character coverage: "
        f"{example['character_coverage']}/"
        f"{example['gold_length']} "
        f"({example['coverage_ratio']:.1%})"
    )

    print("Predicted fragments:")

    for fragment in example["overlapping_predictions"]:
        print(
            f"  [{fragment['start']}:{fragment['end']}] "
            f"{fragment['text']!r}"
        )

    shown += 1

    if shown >= 20:
        break

print("\nTotal fragmented gold entities:", len(fragment_examples))
print("Diagnostic complete. No weights changed.")

ENTITY FRAGMENTATION SUMMARY
LABEL           GOLD   EXACT   NO OVERLAP   ONE INEXACT  MULTI PRED   AVG COVERAGE
APP               91       9           45            27          10        33.86%
RESP              74       2           50            21           1        13.64%
A.COUNSEL         35      11           10             7           7        48.53%
R.COUNSEL         41      11            7            14           9        62.21%
JUDGE             50      25            6            13           6        75.00%
WIT               40      13           21             4           2        39.21%
AUTH             176      26          115            30           5        23.82%
COURT            184     164            7            12           1        93.47%
STAT             195     129            7            36          23        85.15%
PREC             233      54           62            49          68        59.14%
DATE             314     273           13            23           5 

# Phase 2 — Cross-Label Confusion and Boundary Diagnostics

## Best Baseline

- Architecture: Arch4
- Adaptation method: Full fine-tuning
- Selected epoch: 4
- Validation Strict Macro-F1: 0.314728

## Major Cross-Label Confusions

| Predicted label | Gold label | Count |
|---|---|---:|
| APP | RESP | 44 |
| A.COUNSEL | R.COUNSEL | 29 |
| R.COUNSEL | A.COUNSEL | 21 |
| AUTH | PREC | 18 |
| WIT | RESP | 15 |
| RESP | APP | 15 |

The overlap-based diagnostics indicate potential confusion between
party roles, counsel roles, and certain legal entity types.

## Boundary-Related Findings

`PREC` has the largest count of same-label-overlapping false positives
and false negatives. `CASENO` and `STAT` also exhibit substantial
boundary-related errors.

Long citations may be fragmented into smaller predicted entities.
This motivates a focused diagnostic measuring the number of predicted
fragments and the proportion of gold characters they cover.

## Detection Findings

`AUTH` has 121 false negatives with no overlapping prediction,
indicating that missed detections are also an important failure mode.

## Next Step

Quantify per-label fragmentation and character coverage before
choosing a corrective experiment.

The Epoch 4 checkpoint remains the reference baseline. The official
test fold remains untouched.

# Phase 2 — Entity Fragmentation Analysis

## Baseline

- Architecture: Arch4
- Adaptation: Full fine-tuning
- Best checkpoint: Epoch 4
- Validation Strict Macro-F1: 0.314728

## Fragmentation Results

A fragmented gold entity is a gold span overlapped by multiple
same-label predicted spans without an exact match.

| Entity | Fragmented entities | Average character coverage |
|---|---:|---:|
| PREC | 83 | 62.65% |
| CASENO | 30 | 72.59% |
| STAT | 12 | 89.73% |
| R.COUNSEL | 9 | 53.73% |
| AUTH | 6 | 19.20% |
| JUDGE | 6 | 75.18% |
| APP | 5 | 24.51% |
| DATE | 4 | 91.62% |
| A.COUNSEL | 4 | 55.28% |
| COURT | 2 | 91.76% |
| RESP | 2 | 13.53% |
| WIT | 1 | 36.88% |
| **Total** | **164** | — |

The total number of fragmented gold entities is 164 out of 1,585
validation entities, approximately 10.3%.

PREC, CASENO, and STAT account for 125 of the 164 fragmented
entities, approximately 76.2%.

## Interpretation

The examples show that citation-heavy PREC entities may be split
into several disconnected spans. CASENO entities can also be
fragmented, particularly when identifiers contain abbreviations,
punctuation, and multiple numbers.

STAT often has high character coverage despite imperfect boundaries,
suggesting that some errors involve missing prefixes or suffixes.

AUTH exhibits a different issue: 125 of its 176 gold entities have
no overlapping same-label prediction. Missed detection therefore
deserves separate attention from entity fragmentation.

## Limitations

These counts are diagnostic indicators, not proof that every error
is caused by one mechanism. A fragment may also have an incorrect
label, and overlapping spans can interact in complex ways.

No automatic span-merging rules have been applied.

## Next Steps

1. Verify the custom evaluator against the official capstone evaluator.
2. Preserve the Arch4 Epoch 4 checkpoint as the baseline.
3. Proceed with the required architecture and adaptation experiments.
4. Report per-label metrics and fragmentation findings alongside
   Strict Macro-F1.

The official test fold remains untouched.

# Phase 2 — Experiment Tracking

## Completed Experiment

| Property | Result |
|---|---|
| Architecture | Arch4 |
| Adaptation method | Full fine-tuning |
| Best epoch | 4 |
| Strict Macro-Precision | 0.273760 |
| Strict Macro-Recall | 0.383288 |
| **Strict Macro-F1** | **0.314728** |
| Training documents | 56 |
| Validation documents | 14 |
| Official test documents | 35 |

### Best Checkpoint

`/kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt`

### Observations

- Validation Strict Macro-F1 peaked at Epoch 4.
- Epoch 5 had lower training loss but lower validation F1.
- COURT, DATE and STAT were the strongest entity classes.
- APP, RESP, AUTH and PREC require further investigation.

### Remaining Work

1. Evaluate Arch1, Arch2 and Arch3.
2. Run the required LORMA experiment.
3. Evaluate another distinct PEFT method.
4. Run a knowledge-distillation experiment.
5. Compare validation Strict Macro-F1 and per-label metrics.
6. Select the best experiments.
7. Generate final predictions for the official test fold.

In [25]:
# ============================================================
# ARCH1 — ORIGINAL PHASE-1 MODEL DEFINITIONS
# Definitions only. Does NOT start Phase-1 training.
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# 1. Causal self-attention
# ------------------------------------------------------------

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        mask = torch.tril(
            torch.ones(block_size, block_size)
        ).view(1, 1, block_size, block_size)

        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        att = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        att = att.masked_fill(
            self.mask[:, :, :T, :T] == 0,
            float("-inf")
        )

        att = F.softmax(att, dim=-1)

        y = att @ v

        y = (
            y.transpose(1, 2)
             .contiguous()
             .view(B, T, C)
        )

        return self.out_proj(y)


# ------------------------------------------------------------
# 2. Feed-forward network
# ------------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(
            F.gelu(self.fc1(x))
        )


# ------------------------------------------------------------
# 3. Transformer block
# ------------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        block_size
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)

        self.attn = CausalSelfAttention(
            d_model,
            n_heads,
            block_size
        )

        self.norm2 = nn.LayerNorm(d_model)

        self.mlp = FeedForward(
            d_model,
            d_ff
        )

    def forward(self, x):
        x = x + self.attn(
            self.norm1(x)
        )

        x = x + self.mlp(
            self.norm2(x)
        )

        return x


# ------------------------------------------------------------
# 4. Arch1 language-model backbone
# ------------------------------------------------------------

class LegalDecoderLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        block_size,
        d_model,
        n_layers,
        n_heads,
        d_ff
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model

        self.token_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_emb = nn.Embedding(
            block_size,
            d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                n_heads,
                d_ff,
                block_size
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Original Phase-1 tied embeddings
        self.lm_head.weight = self.token_emb.weight

    def forward(self, x):
        B, T = x.shape

        assert T <= self.block_size

        token_vectors = self.token_emb(x)

        positions = torch.arange(
            T,
            device=x.device
        )

        position_vectors = self.pos_emb(positions)

        h = token_vectors + position_vectors

        for block in self.blocks:
            h = block(h)

        h = self.final_norm(h)

        return self.lm_head(h)


# ------------------------------------------------------------
# 5. Original Arch1 constructor
# ------------------------------------------------------------

def build_model(vocab_size, block_size):
    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=768,
        n_layers=8,
        n_heads=8,
        d_ff=3072
    )


print("=" * 65)
print("ARCH1 MODEL DEFINITIONS")
print("=" * 65)

print("LegalDecoderLM defined:", "LegalDecoderLM" in globals())
print("build_model defined:", callable(build_model))

# Verify constructor and parameter count.
_arch1_check = build_model(40000, 1024)

_arch1_params = sum(
    p.numel()
    for p in _arch1_check.parameters()
)

print("Unique model parameters:", f"{_arch1_params:,}")
print("Model dimension:", _arch1_check.d_model)
print("Transformer layers:", len(_arch1_check.blocks))
print("Attention heads:", _arch1_check.blocks[0].attn.n_heads)

assert _arch1_check.d_model == 768
assert len(_arch1_check.blocks) == 8
assert _arch1_check.blocks[0].attn.n_heads == 8
assert 50_000_000 <= _arch1_params <= 300_000_000

del _arch1_check

print("\nARCH1 MODEL DEFINITIONS: PASS")
print("No Phase-1 training was performed.")

ARCH1 MODEL DEFINITIONS
LegalDecoderLM defined: True
build_model defined: True
Unique model parameters: 88,210,944
Model dimension: 768
Transformer layers: 8
Attention heads: 8

ARCH1 MODEL DEFINITIONS: PASS
No Phase-1 training was performed.


# Arch1 Phase-2 — Model Initialization

## Objective

Reconstruct the original Phase-1 Arch1 architecture in the active
Kaggle session without rerunning Phase-1 training.

## Architecture

| Property | Value |
|---|---:|
| Vocabulary size | 40,000 |
| Model dimension | 768 |
| Transformer layers | 8 |
| Attention heads | 8 |
| Feed-forward dimension | 3,072 |
| Maximum sequence length | 1,024 |

## Initialization Procedure

1. Define the original Arch1 model classes.
2. Instantiate the architecture.
3. Verify the architecture dimensions and parameter count.
4. Load the existing Phase-1 checkpoint using strict state-dictionary
   matching.
5. Attach the 25-class BIO classification head.
6. Validate the forward pass before Phase-2 fine-tuning.

## Status

Model-definition status: PASS only after the class-definition
cell completes successfully.

Phase-2 training status: Pending checkpoint and tokenizer preflight.

In [26]:
# ============================================================
# PHASE 2 — ARCH4 + LORMA+ FINE-TUNING
# ============================================================
# Starting weights: Phase-1 Arch4 best.pt
# Adaptation: Identity-inflated multiplicative low-rank update
# Task: Legal Named Entity Recognition
# Classes: 25 BIO classes
#
# Only the LoRMA factors and BIO classification head are trained.
# The original Arch4 weights remain frozen.
#
# Validation: Document-level Strict Macro-F1
# Test fold: NOT USED
# ============================================================

import gc
import os
import json
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset


# ------------------------------------------------------------
# 1. Preconditions and configuration
# ------------------------------------------------------------

assert "Architecture4" in globals(), (
    "Architecture4 definition missing. Run the original Arch4 "
    "model-definition cell first."
)

assert "train_docs" in globals() and "val_docs" in globals()
assert "tokenizer" in globals()
assert "correct_offsets" in globals()
assert "ENTITY_LABELS" in globals()
assert "TAG2ID" in globals() and len(TAG2ID) == 25
assert "ID2TAG" in globals()
assert "evaluate_strict_macro_f1" in globals(), (
    "Run the document-level Strict Macro-F1 evaluator definition first."
)

ARCH4_CKPT = Path(
    "/kaggle/input/models/achiversx/"
    "arch4-hybrid-gqa-201m/transformers/default/1/"
    "arch4_hybrid_gqa_201m/best.pt"
)

DATA_DIR = Path("/kaggle/working/phase2_arch4_data")

TRAIN_PATH = DATA_DIR / "train_windows.pt"
VAL_PATH = DATA_DIR / "validation_windows.pt"

OUT_DIR = Path("/kaggle/working/phase2_arch4_lorma")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = OUT_DIR / "best_lorma_strict_macro_f1.pt"
HISTORY_PATH = OUT_DIR / "training_history.json"

MAX_LEN = 1024
NUM_CLASSES = 25
IGNORE_INDEX = -100

# Fixed baseline configuration for this LORMA experiment.
# This is one experiment, not a hyperparameter search.
LORMA_CFG = {
    "variant": "LoRMA+ identity inflation",
    "rank": 8,
    "alpha": 16.0,
    "dropout": 0.05,
    "epochs": 5,
    "batch_size": 1,
    "grad_accum": 8,
    "adapter_lr": 1e-4,
    "head_lr": 2e-4,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "seed": 42,
    "max_length": MAX_LEN,
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert DEVICE.type == "cuda", (
    "Please enable the Kaggle GPU."
)

assert ARCH4_CKPT.is_file(), (
    f"Arch4 Phase-1 checkpoint not found: {ARCH4_CKPT}"
)

assert TRAIN_PATH.is_file()
assert VAL_PATH.is_file()

random.seed(LORMA_CFG["seed"])
torch.manual_seed(LORMA_CFG["seed"])
torch.cuda.manual_seed_all(LORMA_CFG["seed"])

USE_AMP = True

print("=" * 72)
print("ARCH4 + LORMA+ — PHASE 2")
print("=" * 72)

print("GPU:", torch.cuda.get_device_name(0))
print("Starting checkpoint:", ARCH4_CKPT)
print("Adaptation:", LORMA_CFG["variant"])


# ------------------------------------------------------------
# 2. Load datasets
# ------------------------------------------------------------

train_data = torch.load(
    TRAIN_PATH,
    map_location="cpu",
    weights_only=False
)

val_data = torch.load(
    VAL_PATH,
    map_location="cpu",
    weights_only=False
)

assert train_data["input_ids"].shape[1] == MAX_LEN
assert val_data["input_ids"].shape[1] == MAX_LEN

assert train_data["labels"].shape == train_data["input_ids"].shape
assert val_data["labels"].shape == val_data["input_ids"].shape

assert len(train_docs) == 56
assert len(val_docs) == 14

print("\nTraining windows:", len(train_data["metadata"]))
print("Validation windows:", len(val_data["metadata"]))
print("BIO classes:", NUM_CLASSES)


# ------------------------------------------------------------
# 3. Free the previous Phase-2 model from GPU memory
# ------------------------------------------------------------
# The Epoch-4 full-fine-tuning checkpoint remains saved on disk.

if "backbone" in globals():
    del backbone

if "bio_head" in globals():
    del bio_head

gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# 4. Load original Phase-1 Arch4 weights
# ------------------------------------------------------------

phase1_ckpt = torch.load(
    ARCH4_CKPT,
    map_location="cpu",
    weights_only=False
)

assert "model" in phase1_ckpt
assert "cfg" in phase1_ckpt

ARCH4_CFG = dict(phase1_ckpt["cfg"])

lorma_backbone = Architecture4(ARCH4_CFG)

load_result = lorma_backbone.load_state_dict(
    phase1_ckpt["model"],
    strict=True
)

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

lorma_backbone = lorma_backbone.to(DEVICE).eval()

print("\nPhase-1 checkpoint load: PASS")
print("Checkpoint step:", phase1_ckpt.get("step"))
print("Best validation loss:", phase1_ckpt.get("best_val"))

backbone_params = sum(
    p.numel()
    for p in lorma_backbone.parameters()
)

print("Unique Arch4 parameters:", f"{backbone_params:,}")

assert backbone_params == 201_443_328


# ------------------------------------------------------------
# 5. Multiplicative low-rank adaptation module
# ------------------------------------------------------------
# Identity-inflated multiplicative transformation:
#
#   y0 = W0(x)
#   y  = (I + alpha/r * B @ A) y0
#      = y0 + alpha/r * B(A(y0))
#
# A: output_dim -> rank
# B: rank -> output_dim
#
# B starts at zero, so the adapted projection initially reproduces
# the original projection. The base projection stays frozen.
#
# This is the identity-inflation LoRMA+ formulation, not additive
# LoRA's W0(x) + B(A(x)) update.

class LoRMAPlusLinear(nn.Module):

    def __init__(
        self,
        base_layer,
        rank,
        alpha,
        dropout=0.0
    ):
        super().__init__()

        assert isinstance(base_layer, nn.Linear)
        assert rank > 0

        self.base = base_layer

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        self.rank = rank
        self.alpha = float(alpha)
        self.scale = self.alpha / self.rank

        self.dropout = nn.Dropout(dropout)

        self.A = nn.Linear(
            self.out_features,
            rank,
            bias=False
        )

        self.B = nn.Linear(
            rank,
            self.out_features,
            bias=False
        )

        # Preserve the original base model.
        for p in self.base.parameters():
            p.requires_grad_(False)

        # A has a standard nonzero initialization.
        nn.init.kaiming_uniform_(
            self.A.weight,
            a=math.sqrt(5)
        )

        # Identity inflation at initialization:
        # I + scale * B @ A initially equals I.
        nn.init.zeros_(self.B.weight)

    def forward(self, x):

        # Original frozen projection.
        y0 = self.base(x)

        # Multiplicative identity-inflation transformation.
        update = self.B(
            self.A(
                self.dropout(y0)
            )
        )

        return y0 + self.scale * update


# ------------------------------------------------------------
# 6. Attach LoRMA+ adapters to Arch4 transformer projections
# ------------------------------------------------------------
# The original Phase-1 model is loaded before any modules are wrapped.
# The vocabulary embedding and LM head are not adapted.

for parameter in lorma_backbone.parameters():
    parameter.requires_grad_(False)

wrapped_modules = []

def attach_lorma(parent, attribute, module_name):

    original = getattr(parent, attribute)

    assert isinstance(original, nn.Linear), (
        f"Expected a Linear layer at {module_name}, "
        f"found {type(original)}"
    )

    adapted = LoRMAPlusLinear(
        base_layer=original,
        rank=LORMA_CFG["rank"],
        alpha=LORMA_CFG["alpha"],
        dropout=LORMA_CFG["dropout"]
    )

    setattr(parent, attribute, adapted)

    wrapped_modules.append(module_name)


for layer_index, block in enumerate(lorma_backbone.layers):

    attach_lorma(
        block.attn,
        "qkv_proj",
        f"layers.{layer_index}.attn.qkv_proj"
    )

    attach_lorma(
        block.attn,
        "out_proj",
        f"layers.{layer_index}.attn.out_proj"
    )

    attach_lorma(
        block.ffn,
        "in_proj",
        f"layers.{layer_index}.ffn.in_proj"
    )

    attach_lorma(
        block.ffn,
        "out_proj",
        f"layers.{layer_index}.ffn.out_proj"
    )


lorma_backbone = lorma_backbone.to(DEVICE).eval()

print("\nAdapted projection modules:", len(wrapped_modules))

assert len(wrapped_modules) == 48


# ------------------------------------------------------------
# 7. Verify identity-inflation initialization
# ------------------------------------------------------------
# Zero-initialized B should make the wrapped model initially
# match the original frozen backbone.

probe = train_data["input_ids"][:1].to(DEVICE)

with torch.inference_mode():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        reference_hidden = lorma_backbone.hidden_states(probe).float()

# Since the model was wrapped with zero-initialized B, this
# output should remain numerically unchanged.

with torch.inference_mode():

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        adapted_hidden = lorma_backbone.hidden_states(probe).float()

identity_max_diff = (
    reference_hidden - adapted_hidden
).abs().max().item()

print("Identity-inflation maximum difference:", identity_max_diff)

assert identity_max_diff < 1e-5, (
    "LoRMA+ initialization changed the backbone output unexpectedly."
)

del probe, reference_hidden, adapted_hidden
gc.collect()
torch.cuda.empty_cache()

print("Identity-inflation check: PASS")


# ------------------------------------------------------------
# 8. Add fresh BIO classification head
# ------------------------------------------------------------

torch.manual_seed(LORMA_CFG["seed"])

backbone = lorma_backbone

bio_head = nn.Linear(
    ARCH4_CFG["D_MODEL"],
    NUM_CLASSES
).to(DEVICE)

assert bio_head.in_features == 1024
assert bio_head.out_features == NUM_CLASSES

# Verify that only LoRMA factors are trainable in the backbone.
adapter_named_params = [
    (name, p)
    for name, p in backbone.named_parameters()
    if p.requires_grad
]

assert adapter_named_params
assert all(
    (".A." in name or ".B." in name)
    for name, _ in adapter_named_params
), "Unexpected trainable backbone parameter detected."

adapter_params = [
    p for _, p in adapter_named_params
]

head_params = list(bio_head.parameters())

adapter_param_count = sum(
    p.numel() for p in adapter_params
)

head_param_count = sum(
    p.numel() for p in head_params
)

print("\nTrainable parameter breakdown:")
print("LoRMA+ parameters:", f"{adapter_param_count:,}")
print("BIO head parameters:", f"{head_param_count:,}")
print(
    "Total trainable:",
    f"{adapter_param_count + head_param_count:,}"
)

print(
    "Trainable percentage:",
    f"{100 * (adapter_param_count + head_param_count) / backbone_params:.3f}%"
)


# ------------------------------------------------------------
# 9. Prepare optimizer and training loader
# ------------------------------------------------------------

train_dataset = TensorDataset(
    train_data["input_ids"],
    train_data["labels"]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=LORMA_CFG["batch_size"],
    shuffle=True,
    generator=torch.Generator().manual_seed(
        LORMA_CFG["seed"]
    ),
    num_workers=0,
    pin_memory=True
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": adapter_params,
            "lr": LORMA_CFG["adapter_lr"],
            "weight_decay": 0.0
        },
        {
            "params": head_params,
            "lr": LORMA_CFG["head_lr"],
            "weight_decay": LORMA_CFG["weight_decay"]
        }
    ]
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

DEVICE = torch.device("cuda")
USE_AMP = True


# ------------------------------------------------------------
# 10. Save only trainable adapter and head weights
# ------------------------------------------------------------
# The Phase-1 backbone weights remain available in the original
# checkpoint. This saves a compact experiment checkpoint.

def save_lorma_checkpoint(epoch, metrics, train_loss):

    adapter_state = {
        name: p.detach().cpu().clone()
        for name, p in backbone.named_parameters()
        if p.requires_grad
    }

    head_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in bio_head.state_dict().items()
    }

    payload = {
        "experiment": "Arch4 + LoRMA+ identity inflation",
        "epoch": epoch,
        "phase1_checkpoint": str(ARCH4_CKPT),
        "adapter_state": adapter_state,
        "bio_head_state": head_state,
        "metrics": metrics,
        "train_loss": train_loss,
        "config": LORMA_CFG,
        "architecture_config": ARCH4_CFG,
        "tag2id": TAG2ID,
        "id2tag": ID2TAG,
        "wrapped_modules": wrapped_modules,
        "trainable_parameters": (
            adapter_param_count + head_param_count
        ),
    }

    temporary_path = BEST_CKPT.with_suffix(".tmp")

    torch.save(
        payload,
        temporary_path
    )

    os.replace(
        temporary_path,
        BEST_CKPT
    )


# ------------------------------------------------------------
# 11. Training and validation
# ------------------------------------------------------------

best_macro_f1 = -1.0
history = []

print("\n" + "=" * 72)
print("STARTING ARCH4 + LORMA+ TRAINING")
print("=" * 72)

for epoch in range(1, LORMA_CFG["epochs"] + 1):

    backbone.train()
    bio_head.train()

    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    batches_seen = 0

    for batch_idx, (input_ids, labels) in enumerate(train_loader):

        input_ids = input_ids.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )

        group_start = (
            batch_idx // LORMA_CFG["grad_accum"]
        ) * LORMA_CFG["grad_accum"]

        group_size = min(
            LORMA_CFG["grad_accum"],
            len(train_loader) - group_start
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            hidden = backbone.hidden_states(input_ids)

            logits = bio_head(hidden)

            loss = F.cross_entropy(
                logits.reshape(-1, NUM_CLASSES).float(),
                labels.reshape(-1),
                ignore_index=IGNORE_INDEX
            )

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at epoch {epoch}, batch {batch_idx}"
            )

        running_loss += float(loss.detach())
        batches_seen += 1

        scaler.scale(
            loss / group_size
        ).backward()

        update_now = (
            (batch_idx + 1) % LORMA_CFG["grad_accum"] == 0
            or (batch_idx + 1) == len(train_loader)
        )

        if update_now:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                adapter_params + head_params,
                LORMA_CFG["max_grad_norm"]
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        if (
            batch_idx == 0
            or (batch_idx + 1) % 50 == 0
            or (batch_idx + 1) == len(train_loader)
        ):
            print(
                f"Epoch {epoch}/{LORMA_CFG['epochs']} | "
                f"Batch {batch_idx + 1}/{len(train_loader)} | "
                f"Loss {float(loss.detach()):.4f}"
            )

        del input_ids, labels, hidden, logits, loss

    mean_train_loss = running_loss / max(batches_seen, 1)

    # The validated evaluator reconstructs full documents and
    # calculates exact character-span metrics.
    metrics = evaluate_strict_macro_f1()

    macro_f1 = metrics["strict_macro_f1"]

    epoch_record = {
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "strict_macro_precision": metrics["strict_macro_precision"],
        "strict_macro_recall": metrics["strict_macro_recall"],
        "strict_macro_f1": macro_f1,
        "gold_entities": metrics["gold_entities"],
        "predicted_entities": metrics["predicted_entities"],
        "exact_matches": metrics["exact_matches"],
    }

    history.append(epoch_record)

    print("\n" + "=" * 72)
    print(f"LORMA EPOCH {epoch} SUMMARY")
    print("=" * 72)

    print(f"Training loss: {mean_train_loss:.6f}")
    print(
        "Validation Macro-Precision:",
        f"{metrics['strict_macro_precision']:.6f}"
    )
    print(
        "Validation Macro-Recall:",
        f"{metrics['strict_macro_recall']:.6f}"
    )
    print(
        "Validation Strict Macro-F1:",
        f"{macro_f1:.6f}"
    )

    print("Gold entities:", metrics["gold_entities"])
    print("Predicted entities:", metrics["predicted_entities"])
    print("Exact matches:", metrics["exact_matches"])

    if macro_f1 > best_macro_f1:

        best_macro_f1 = macro_f1

        save_lorma_checkpoint(
            epoch,
            metrics,
            mean_train_loss
        )

        print("\nNEW BEST LORMA CHECKPOINT SAVED")
        print(BEST_CKPT)

    with open(
        HISTORY_PATH,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(history, f, indent=2)

    print("=" * 72 + "\n")


# ------------------------------------------------------------
# 12. Restore best LoRMA checkpoint
# ------------------------------------------------------------

saved = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=False
)

current_trainable = {
    name: p
    for name, p in backbone.named_parameters()
    if p.requires_grad
}

assert set(current_trainable) == set(saved["adapter_state"])

with torch.no_grad():

    for name, param in current_trainable.items():

        param.copy_(
            saved["adapter_state"][name].to(
                device=param.device,
                dtype=param.dtype
            )
        )

bio_head.load_state_dict(
    saved["bio_head_state"],
    strict=True
)

backbone.eval()
bio_head.eval()

# Final evaluation from restored best weights.
best_metrics = evaluate_strict_macro_f1()

backbone.eval()
bio_head.eval()


# ------------------------------------------------------------
# 13. Final per-label validation report
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("ARCH4 + LORMA+ — BEST VALIDATION CHECKPOINT")
print("=" * 88)

print("Best epoch:", saved["epoch"])
print("Trainable parameters:", saved["trainable_parameters"])
print("Checkpoint:", BEST_CKPT)

print(
    "Strict Macro-Precision:",
    f"{best_metrics['strict_macro_precision']:.6f}"
)

print(
    "Strict Macro-Recall:",
    f"{best_metrics['strict_macro_recall']:.6f}"
)

print(
    "Strict Macro-F1:",
    f"{best_metrics['strict_macro_f1']:.6f}"
)

print("\nPer-label metrics:")
print(
    f"{'LABEL':<12}"
    f"{'GOLD':>8}"
    f"{'PRED':>8}"
    f"{'TP':>8}"
    f"{'FP':>8}"
    f"{'FN':>8}"
    f"{'PRECISION':>12}"
    f"{'RECALL':>10}"
    f"{'F1':>10}"
)

for label in ENTITY_LABELS:

    m = best_metrics["per_label"][label]

    print(
        f"{label:<12}"
        f"{m['gold']:>8}"
        f"{m['predicted']:>8}"
        f"{m['tp']:>8}"
        f"{m['fp']:>8}"
        f"{m['fn']:>8}"
        f"{m['precision']:>12.4f}"
        f"{m['recall']:>10.4f}"
        f"{m['f1']:>10.4f}"
    )

print("\n" + "=" * 72)
print("ARCH4 + LORMA+ EXPERIMENT COMPLETE")
print("=" * 72)

print("Best validation Strict Macro-F1:", f"{best_metrics['strict_macro_f1']:.6f}")
print("Best checkpoint:", BEST_CKPT)
print("Training history:", HISTORY_PATH)
print("Official test fold used: NO")
print("prediction.json generated: NO")

ARCH4 + LORMA+ — PHASE 2
GPU: Tesla T4
Starting checkpoint: /kaggle/input/models/achiversx/arch4-hybrid-gqa-201m/transformers/default/1/arch4_hybrid_gqa_201m/best.pt
Adaptation: LoRMA+ identity inflation

Training windows: 502
Validation windows: 105
BIO classes: 25

Phase-1 checkpoint load: PASS
Checkpoint step: 10000
Best validation loss: 2.475979009747505
Unique Arch4 parameters: 201,443,328

Adapted projection modules: 48
Identity-inflation maximum difference: 0.0
Identity-inflation check: PASS

Trainable parameter breakdown:
LoRMA+ parameters: 2,064,384
BIO head parameters: 25,625
Total trainable: 2,090,009
Trainable percentage: 1.038%

STARTING ARCH4 + LORMA+ TRAINING
Epoch 1/5 | Batch 1/502 | Loss 3.3470
Epoch 1/5 | Batch 50/502 | Loss 3.4121
Epoch 1/5 | Batch 100/502 | Loss 1.6947
Epoch 1/5 | Batch 150/502 | Loss 0.6308
Epoch 1/5 | Batch 200/502 | Loss 0.5055
Epoch 1/5 | Batch 250/502 | Loss 1.0199
Epoch 1/5 | Batch 300/502 | Loss 0.5706
Epoch 1/5 | Batch 350/502 | Loss 0.1347


# Phase 2 — Arch4 + LoRMA+ Fine-Tuning

## Objective

Evaluate identity-inflated multiplicative low-rank adaptation
(LoRMA+) for Legal Named Entity Recognition using the Phase-1
Arch4 language model.

## Initialization

The experiment starts from the original Phase-1 Arch4 checkpoint.
The full-fine-tuning Epoch 4 checkpoint is retained separately
as the baseline and is not used to initialize this experiment.

## Adaptation

The transformer projection layers receive multiplicative
low-rank adapters. The original Arch4 weights remain frozen.

The trainable parameters are:

- LoRMA+ low-rank factors
- The 25-class BIO classification head

The update uses an identity-inflated multiplicative formulation,
so the adapter initially preserves the original projection.

## Dataset

| Partition | Documents | Windows |
|---|---:|---:|
| Training | 56 | 502 |
| Validation | 14 | 105 |
| Official test | 35 | Not used |

## Evaluation

Validation is performed after each epoch using document-level
exact character-span matching.

Primary metric: Strict Macro-F1.

Secondary metrics: Strict Macro-Precision, Strict Macro-Recall,
and per-label F1.

The checkpoint with the highest validation Strict Macro-F1 is
selected.

## Artifacts

Best checkpoint:

`/kaggle/working/phase2_arch4_lorma/best_lorma_strict_macro_f1.pt`

Training history:

`/kaggle/working/phase2_arch4_lorma/training_history.json`

## Status

Pending completion of the fine-tuning run.

The official test fold remains untouched. No `prediction.json`
is generated during training or validation.

# Phase 2 — Arch4 Full Fine-Tuning vs LoRMA+

## Experiment 1: Full Fine-Tuning

| Metric | Value |
|---|---:|
| Best epoch | 4 |
| Strict Macro-Precision | 0.273760 |
| Strict Macro-Recall | 0.383288 |
| Strict Macro-F1 | 0.314728 |

Checkpoint:

`/kaggle/working/phase2_arch4_baseline/best_strict_macro_f1.pt`

## Experiment 2: LoRMA+

| Metric | Value |
|---|---:|
| Best epoch | 5 |
| Trainable parameters | 2,090,009 |
| Trainable percentage | 1.038% |
| Strict Macro-Precision | 0.252559 |
| Strict Macro-Recall | 0.337685 |
| Strict Macro-F1 | 0.285182 |

Checkpoint:

`/kaggle/working/phase2_arch4_lorma/best_lorma_strict_macro_f1.pt`

## Comparison

Full fine-tuning achieved the higher validation Strict Macro-F1.

LoRMA+ updates substantially fewer parameters, but its current
validation performance is lower.

Per-label results indicate that the methods have different strengths.
LoRMA+ performs better on R.COUNSEL, AUTH, COURT, PREC and CASENO,
while full fine-tuning performs better on several other labels,
particularly STAT and JUDGE.

## Interpretation

These results represent two independent adaptation experiments
initialized from the original Phase-1 Arch4 checkpoint.

The full-fine-tuning checkpoint remains the stronger overall
validation baseline.

The official test fold remains untouched.








